# 01_syntgen: Synthetic patient generator
Run cells from top to bottom. After **Restart**, use **Run All**.

## Setup

In [1]:
from datetime import date
import numpy as np, pandas as pd
import ipytest
# raise_on_error: a test cell that fails or collects no tests stops the run.
# addopts without the default "-q", so each cell shows "N passed"; no colour codes in saved outputs.
ipytest.autoconfig(raise_on_error=True, addopts=["--color=no", "-p", "no:cacheprovider",
                                                 "-W", "ignore::pytest.PytestAssertRewriteWarning"])


CFG = {
    "seed": 20261101,
    "n_patients": 5000,
    "window": {"start_year": 2021, "end_year": 2026},
    "visits": {"max": 8},
    "visit_gap_months": {"Low": 12, "Moderate": 4, "High": 1.5},
    "family_multiplier": 2.0,
    "relatives": {"confirmed_share": 0.70, "unknown_onset_share": 0.20},
    "paths": {"healthy": 0.50, "progressor": 0.25, "already_ill": 0.15, "improver": 0.10},
    "fhrs": {
        "R": {1: 0.50, 2: 0.25, 3: 0.125},
        "S": {"confirmed": 1.0, "not_sure": 0.5},
        "cluster_bonus": 1.0,
        "penalties": {
            "CVD": {"secondhand_smoke": 0.15, "high_salt_diet": 0.15},
            "T2D": {"processed_diet": 0.15, "low_home_activity": 0.15},
            "CRD": {"secondhand_smoke": 0.15, "indoor_mold": 0.15, "coal_heating": 0.15},
            "NEURO": {},
        },
    },
    "drift_per_year": {"hba1c": {"healthy": 0.02, "progressor": 0.20, "improver": -0.10}},
    "missing": {"optional_lab": 0.70, "hospital_only": 0.93},
    "label_window_years": 5,
    "gbd_file": "data/gbd_2023_ph_rates.csv",
    "deaths_file": "data/psa_deaths_2024.csv",
    "pop_file": "data/psa_population_projection.csv",
    "death_year": "2024",
    # Tuned so the share with each disease matches 02_synthval Cell 3 targets (see Part 0 note):
    # CVD, asthma, COPD, T2D (all, diagnosed + undiagnosed) -> GBD 2023 prevalence by age/sex;
    # diagnosed T2D -> NOH 2023-2028 (7.8-7.9%) through AD["t2d_diagnosed_share"]
    "prevalence_factor": {"CVD": 1.75, "T2D": 1.37, "asthma": 1.55, "COPD": 1.95},
    "lifetime_target": {
        "Major depressive disorder": {"Female": 0.148, "Male": 0.074},
        "Anxiety disorders": {"Female": 0.016, "Male": 0.016},
        "Asthma": {"Female": 0.10, "Male": 0.10},
    },
    "checks": {
        "hypertension": (0.17, 0.23),
        "current_smoker": 0.204,
        "current_depression": 0.033,
    },
}
RNG = np.random.default_rng(CFG["seed"])
print("Setup OK")

Setup OK


Part 0: Prepare disease rates

In [2]:
# Cell 2 - CODE
AGE_GROUPS = [f"{a}-{a + 4} years" for a in range(15, 95, 5)] + ["95+ years"]

def load_rates(path):
    g = pd.read_csv(path)
    g = g[g["age_name"].isin(AGE_GROUPS) & (g["metric_name"] == "Rate")]
    wide = g.pivot_table(index=["cause_name", "sex_name", "age_name"], 
                         columns="measure_name", values="val").reset_index()
    wide["age_lo"] = wide["age_name"].str.extract(r"^(\d+)").astype(int)
    return pd.DataFrame({
        "disease": wide["cause_name"], "sex": wide["sex_name"], "age_lo": wide["age_lo"],
        "incidence": wide["Incidence"] / 100_000,
        "prevalence": wide["Prevalence"] / 100_000,
    })

def yearly_rate(rates, disease, age, sex, factor=1.0):
    if age < 15: return 0.0
    lo = min(95, age - age % 5)
    row = rates[(rates["disease"] == disease) & (rates["sex"] == sex) & (rates["age_lo"] == lo)]
    return float(row["incidence"].iloc[0]) * factor

def lifetime_risk(rates, disease, sex, factor=1.0, start=15, end=80):
    free = 1.0
    for age in range(start, end):
        free *= 1 - min(yearly_rate(rates, disease, age, sex, factor), 0.99)
    return 1 - free

def calibrate_factor(rates, disease, sex, target, start=15, end=80):
    lo, hi = 0.0, 1.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if lifetime_risk(rates, disease, sex, mid, start, end) < target:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

In [3]:
%%ipytest -q
# Cell 1 - TEST
import pytest

RAW = pd.read_csv(CFG["gbd_file"])
R = load_rates(CFG["gbd_file"])

def test_keeps_only_the_17_five_year_age_groups():
    assert sorted(R["age_lo"].unique()) == list(range(15, 95, 5)) + [95]

def test_has_7_diseases_2_sexes():
    assert R["disease"].nunique() == 7 and set(R["sex"]) == {"Male", "Female"}
    assert len(R) == 7 * 17 * 2

def test_rate_is_chance_per_person_per_year():
    raw = RAW.query("measure_name == 'Incidence' and cause_name == 'Diabetes mellitus type 2' "
                    "and sex_name == 'Male' and age_name == '50-54 years'")["val"].item()
    assert yearly_rate(R, "Diabetes mellitus type 2", 52, "Male") == pytest.approx(raw / 100_000)

def test_rate_is_zero_under_15():
    assert yearly_rate(R, "Stroke", 10, "Female") == 0.0

def test_age_100_uses_95_plus_group():
    raw = RAW.query("measure_name == 'Incidence' and cause_name == 'Stroke' "
                    "and sex_name == 'Female' and age_name == '95+ years'")["val"].item()
    assert yearly_rate(R, "Stroke", 100, "Female") == pytest.approx(raw / 100_000)

def test_factor_scales_the_rate():
    base = yearly_rate(R, "Asthma", 40, "Male")
    assert yearly_rate(R, "Asthma", 40, "Male", factor=0.25) == pytest.approx(base * 0.25)

def test_raw_depression_rates_give_unrealistic_lifetime_risk():
    assert lifetime_risk(R, "Major depressive disorder", "Female") > 0.80

def test_calibration_hits_the_target_lifetime_risk():
    k = calibrate_factor(R, "Major depressive disorder", "Female", target=0.15)
    assert 0 < k < 1
    assert lifetime_risk(R, "Major depressive disorder", "Female", factor=k) == pytest.approx(0.15, abs=0.001)

.

.

.

.

.

.

.

.

                                                                                     [100%]


8 passed in 1.55s


In [4]:
# Cell 3 - RUN
APP_NAME = {"Major depressive disorder": "depression", "Anxiety disorders": "GAD",
            "Asthma": "asthma", "Chronic obstructive pulmonary disease": "COPD",
            "Diabetes mellitus type 2": "T2D"}

RATES = load_rates(CFG["gbd_file"])
RATES["disease"] = RATES["disease"].replace(APP_NAME)

# CVD = ischemic heart disease OR stroke (chance of at least one)
ihd = RATES[RATES["disease"] == "Ischemic heart disease"].set_index(["sex", "age_lo"])
stk = RATES[RATES["disease"] == "Stroke"].set_index(["sex", "age_lo"])
cvd = pd.DataFrame({"incidence": 1 - (1 - ihd["incidence"]) * (1 - stk["incidence"]),
                    "prevalence": ihd["prevalence"] + stk["prevalence"]}).reset_index()
cvd["disease"] = "CVD"
RATES = pd.concat([RATES, cvd[RATES.columns]], ignore_index=True)

TARGET = {APP_NAME.get(d, d): t for d, t in CFG["lifetime_target"].items()}
FACTOR = {(d, s): calibrate_factor(RATES, d, s, t)
          for d, by_sex in TARGET.items() for s, t in by_sex.items()}

# fast lookup: a dict is ~1000x faster than filtering the DataFrame each call
LOOKUP = {(d, s, a): i for d, s, a, i in RATES[["disease", "sex", "age_lo", "incidence"]].itertuples(index=False)}

SEX = {"M": "Male", "F": "Female", "Male": "Male", "Female": "Female"}

def rate_fn(disease, age, sex):
    sex = SEX[sex]
    if age < 15:
        return 0.0
    return (LOOKUP[(disease, sex, min(95, age - age % 5))] * FACTOR.get((disease, sex), 1.0)
            * CFG.get("prevalence_factor", {}).get(disease, 1.0))

print("Factors:", FACTOR)
print(sorted(RATES["disease"].unique()))

Factors: {('depression', 'Female'): 0.07974342611145155, ('depression', 'Male'): 0.05495744013565233, ('GAD', 'Female'): 0.05582501169309552, ('GAD', 'Male'): 0.08667907839000985, ('asthma', 'Female'): 0.22618939865371007, ('asthma', 'Male'): 0.22759915460694619}
['COPD', 'CVD', 'GAD', 'Ischemic heart disease', 'Stroke', 'T2D', 'asthma', 'depression']


In [5]:
%%ipytest -q
# Cell 4 - TEST for the RUN cell (app disease names, CVD, sex codes)
import pytest

def test_rate_fn_knows_every_app_disease_name():
    for d in ["CVD", "T2D", "asthma", "COPD", "depression", "GAD"]:
        for s in ("Male", "Female"):
            assert rate_fn(d, 50, s) > 0

def test_cvd_rate_combines_ischemic_heart_disease_and_stroke():
    ihd = yearly_rate(RATES, "Ischemic heart disease", 60, "Male")
    stroke = yearly_rate(RATES, "Stroke", 60, "Male")
    assert rate_fn("CVD", 60, "Male") == pytest.approx(
        (1 - (1 - ihd) * (1 - stroke)) * CFG["prevalence_factor"].get("CVD", 1.0))

def test_asthma_calibration_factor_is_used():
    raw = yearly_rate(RATES, "asthma", 40, "Female")
    assert rate_fn("asthma", 40, "Female") == pytest.approx(
        raw * FACTOR[("asthma", "Female")] * CFG["prevalence_factor"].get("asthma", 1.0))

def test_rate_fn_is_zero_under_15():
    assert rate_fn("T2D", 10, "Male") == 0.0

def test_rate_fn_accepts_short_sex_codes_used_by_fhrs():
    assert rate_fn("T2D", 50, "F") == rate_fn("T2D", 50, "Female")

def test_prevalence_factor_scales_the_disease_rate():
    import copy
    base = rate_fn("COPD", 60, "Male")
    old = copy.deepcopy(CFG.get("prevalence_factor", {}))
    try:
        CFG["prevalence_factor"] = {**old, "COPD": 3.0}
        assert rate_fn("COPD", 60, "Male") == pytest.approx(base / old.get("COPD", 1.0) * 3.0)
    finally:
        CFG["prevalence_factor"] = old

def test_disease_without_prevalence_factor_is_unchanged():
    assert rate_fn("GAD", 40, "Female") == pytest.approx(
        yearly_rate(RATES, "GAD", 40, "Female") * FACTOR[("GAD", "Female")])

.

.

.

.

.

.

.

                                                                                      [100%]


7 passed in 0.01s


Part 0b: Prepare death rates

In [6]:
# Cell 2 - CODE
def load_death_rates(deaths_path, pop_path, year="2024"):
    d = pd.read_csv(deaths_path, skiprows=1)
    d["Age Group"] = d["Age Group"].replace({"Under 1": "0-4", "1-4": "0-4", "85 and over": "85+"})
    d = d[~d["Age Group"].isin(["Total", "Not Stated"])]
    d = d.groupby("Age Group", sort=False)[["Male", "Female"]].sum()
    
    p = pd.read_csv(pop_path, skiprows=1)
    rows = []
    for sex in ("Male", "Female"):
        pop = p[p["Sex"] == sex].set_index("Age Group")[year] * 1000
        for ag in d.index:
            lo = 85 if ag == "85+" else int(ag.split("-")[0])
            rows.append({"sex": sex, "age_lo": lo, "rate": d.loc[ag, sex] / pop[ag]})
    return pd.DataFrame(rows)

_DR_LOOKUP = {}

def death_rate(dr, age, sex):
    sex = {"M": "Male", "F": "Female"}.get(sex, sex)
    lo = min(85, age - age % 5)
    if id(dr) not in _DR_LOOKUP:          # fast lookup, built once per table
        _DR_LOOKUP[id(dr)] = {(s, a): r for s, a, r in dr[["sex", "age_lo", "rate"]].itertuples(index=False)}
    return float(_DR_LOOKUP[id(dr)][(sex, lo)])

def life_expectancy(dr, sex):
    m = dr[dr["sex"] == sex].sort_values("age_lo")["rate"].tolist()
    alive, years = 1.0, 0.0
    for mx in m[:-1]:
        q = 5 * mx / (1 + 2.5 * mx)
        years += 5 * alive * (1 - q / 2)
        alive *= 1 - q
    return years + alive / m[-1]

def draw_death_year(birth_year, sex, rng, dr, end_year=2026):
    for year in range(birth_year, end_year + 1):
        if rng.random() < death_rate(dr, year - birth_year, sex):
            return year
    return None   # still alive

In [7]:
%%ipytest -q
# Cell 1 - TEST
import pytest

DR = load_death_rates(CFG["deaths_file"], CFG["pop_file"], CFG["death_year"])

def test_population_in_thousands_is_converted():
    assert death_rate(DR, 62, "Male") == pytest.approx(42489 / 1_807_000)

def test_under_1_and_1_to_4_are_merged_into_0_to_4():
    assert death_rate(DR, 2, "Male") == pytest.approx((11714 + 3320) / 4_606_000)

def test_age_90_uses_85_plus_group():
    assert death_rate(DR, 90, "Female") == pytest.approx(53786 / 284_000)

def test_rates_rise_with_age_after_30():
    rates = [death_rate(DR, a, "Female") for a in range(30, 90, 5)]
    assert rates == sorted(rates)

def test_life_expectancy_is_believable():
    assert 65 < life_expectancy(DR, "Male") < 75
    assert 70 < life_expectancy(DR, "Female") < 82
    assert life_expectancy(DR, "Female") > life_expectancy(DR, "Male")

def test_death_year_is_after_birth_and_old_people_are_dead():
    rng = np.random.default_rng(3)
    years = [draw_death_year(1900, "Female", rng, DR) for _ in range(200)]
    assert all(y is not None and 1900 <= y <= 2026 for y in years)

def test_young_person_is_usually_alive():
    rng = np.random.default_rng(3)
    alive = [draw_death_year(2000, "Female", rng, DR) is None for _ in range(500)]
    assert np.mean(alive) > 0.9

def test_death_draw_accepts_short_sex_codes():
    assert draw_death_year(1900, "M", np.random.default_rng(1), DR) is not None

.

.

.

.

.

.

.

.

                                                                                     [100%]


8 passed in 0.03s


In [8]:
# Cell 3 - RUN
DR = load_death_rates(CFG["deaths_file"], CFG["pop_file"], CFG["death_year"])
print("Male Life Expectancy:", life_expectancy(DR, "Male"))
print("Female Life Expectancy:", life_expectancy(DR, "Female"))
print(DR.head())

Male Life Expectancy: 68.93999021366251
Female Life Expectancy: 75.28947913987874
    sex  age_lo      rate
0  Male       0  0.003264
1  Male       5  0.000423
2  Male      10  0.000482
3  Male      15  0.000974
4  Male      20  0.001440


Part 0c: Family size

In [9]:
# Cell 2 - CODE
TFR = {1993: 4.1, 1998: 3.7, 2003: 3.5, 2008: 3.3, 2013: 3.0, 2017: 2.7, 2022: 1.9}

def mean_siblings(birth_year):
    tfr = np.interp(birth_year, list(TFR), list(TFR.values()))
    return float(tfr) - 1

In [10]:
%%ipytest -q
# Cell 1 - TEST
import pytest

def test_mean_siblings_uses_ndhs_tfr_minus_one():
    assert mean_siblings(1993) == pytest.approx(3.1)

def test_mean_siblings_interpolates_between_surveys():
    assert mean_siblings(2010) == pytest.approx(3.3 + (3.0 - 3.3) * 2 / 5 - 1)

def test_births_before_1993_use_the_1993_value():
    assert mean_siblings(1965) == pytest.approx(3.1)

def test_sourced_targets_can_be_reached():
    for disease, by_sex in TARGET.items():
        for sex, target in by_sex.items():
            k = calibrate_factor(RATES, disease, sex, target)
            assert lifetime_risk(RATES, disease, sex, factor=k) == pytest.approx(target, abs=0.001)

.

.

.

.

                                                                                         [100%]


4 passed in 8.90s


In [11]:
# Cell 3 - RUN
print("Mean siblings born in 1960:", mean_siblings(1960))
print("Mean siblings born in 2022:", mean_siblings(2022))

Mean siblings born in 1960: 3.0999999999999996
Mean siblings born in 2022: 0.8999999999999999


Part A: FHRS

In [12]:
# Cell 2 - CODE
DOMAIN = {"CVD": ["CVD"], "T2D": ["T2D"], "CRD": ["asthma", "COPD"], "NEURO": ["depression", "GAD"]}

def onset_multiplier(disease, age, sex):
    if age is None: return 1.0
    if disease == "CVD":
        if age < 40: return 3.0
        if sex == "M": return 2.0 if age < 55 else 1.0 if age < 65 else 0.75
        return 2.0 if age < 65 else 1.0 if age < 75 else 0.75
    if disease in ("T2D", "COPD"):
        return 3.0 if age < 40 else 2.0 if age < 50 else 1.0 if age < 65 else 0.75
    if disease == "asthma":
        return 3.0 if age < 12 else 2.0 if age <= 25 else 1.0 if age <= 40 else 0.75
    if disease == "depression":
        return 3.0 if age < 18 else 2.0 if age <= 24 else 1.0 if age <= 40 else 0.75
    return 1.0

def fhrs(relatives, exposures, domain, on_date, cfg):
    R, S = cfg["fhrs"]["R"], cfg["fhrs"]["S"]
    bio, confirmed_first = 0.0, {}
    
    for r in relatives:
        for c in r["conditions"]:
            if c["disease"] in DOMAIN[domain] and c["year"] <= on_date.year:
                s = S["confirmed"] if c["confirmed"] else S["not_sure"]
                bio += R[r["degree"]] * onset_multiplier(c["disease"], c["age"], r["sex"]) * s
                if r["degree"] == 1 and c["confirmed"]:
                    confirmed_first[c["disease"]] = confirmed_first.get(c["disease"], 0) + 1
                break
                
    if any(n >= 2 for n in confirmed_first.values()):
        bio += cfg["fhrs"]["cluster_bonus"]
        
    E = 1.0 + sum(v for k, v in cfg["fhrs"]["penalties"][domain].items() if exposures.get(k))
    return round(bio * E, 2)

def band(score):
    if score == 0: return 0
    if score <= 1.75: return 1
    if score <= 3.50: return 2
    if score <= 6.00: return 3
    return 4

In [13]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def rel(degree, sex, *conds):
    return {"degree": degree, "sex": sex, 
            "conditions": [{"disease": d, "year": y, "age": a, "confirmed": c} for d, y, a, c in conds]}

def test_cvd_onset_male_48_is_early():
    assert onset_multiplier("CVD", 48, "M") == 2.0

def test_cvd_onset_female_60_is_still_early():
    assert onset_multiplier("CVD", 60, "F") == 2.0

def test_unknown_onset_counts_as_typical():
    assert onset_multiplier("T2D", None, "F") == 1.0

def test_john_example_scores_4_17():
    relatives = [rel(1, "M", ("CVD", 1998, 48, True)), rel(1, "M", ("CVD", 2016, 38, True)), 
                 rel(2, "M", ("CVD", 2008, 60, False))]
    assert fhrs(relatives, {"secondhand_smoke": True}, "CVD", date(2026, 1, 1), CFG) == 4.17

def test_relative_diagnosed_after_visit_is_not_counted():
    assert fhrs([rel(1, "F", ("T2D", 2024, 64, True))], {}, "T2D", date(2023, 6, 1), CFG) == 0.0

def test_relative_diagnosed_before_visit_is_counted():
    assert fhrs([rel(1, "F", ("T2D", 2024, 64, True))], {}, "T2D", date(2025, 6, 1), CFG) == 0.5

def test_cluster_bonus_needs_two_confirmed_first_degree():
    r = [rel(1, "F", ("T2D", 2000, 55, True)), rel(1, "M", ("T2D", 2001, 55, False))]
    assert fhrs(r, {}, "T2D", date(2026, 1, 1), CFG) == 0.75

def test_crd_counts_asthma_and_copd_without_cluster_bonus():
    r = [rel(1, "F", ("asthma", 1990, 30, True)), rel(1, "M", ("COPD", 2010, 70, True))]
    assert fhrs(r, {}, "CRD", date(2026, 1, 1), CFG) == 0.88

def test_cluster_bonus_when_two_confirmed_have_same_disease():
    r = [rel(1, "F", ("asthma", 1990, 30, True)), rel(1, "M", ("asthma", 1995, 35, True))]
    assert fhrs(r, {}, "CRD", date(2026, 1, 1), CFG) == 2.0

def test_neuro_ignores_household_exposures():
    r = [rel(1, "F", ("depression", 2000, 30, True))]
    assert fhrs(r, {"secondhand_smoke": True}, "NEURO", date(2026, 1, 1), CFG) == 0.5

def test_band_edges_match_guide():
    assert [band(s) for s in [0, 0.05, 1.75, 1.76, 3.50, 3.51, 6.00, 6.01]] == [0, 1, 1, 2, 2, 3, 3, 4]

.

.

.

.

.

.

... (3 outputs left out)

.

.

                                                                                  [100%]


11 passed in 0.01s


In [14]:
# Cell 3 - RUN
johns_relatives = [
    {"degree": 1, "sex": "M", "conditions": [{"disease": "CVD", "year": 1998, "age": 48, "confirmed": True}]},
    {"degree": 1, "sex": "M", "conditions": [{"disease": "CVD", "year": 2016, "age": 38, "confirmed": True}]},
    {"degree": 2, "sex": "M", "conditions": [{"disease": "CVD", "year": 2008, "age": 60, "confirmed": False}]}
]
score = fhrs(johns_relatives, {"secondhand_smoke": True}, "CVD", date(2026, 1, 1), CFG)
print(f"John's FHRS Score: {score}")
print(f"John's Band: {band(score)}")

John's FHRS Score: 4.17
John's Band: 3


Part B: Relatives' diseases

In [15]:
# Cell 2 - CODE
def simulate_family_diseases(relatives, cfg, rng, rate_fn, diseases, end_year=2026):
    first = min(r["birth_year"] for r in relatives)
    for year in range(first, end_year + 1):
        for r in relatives:
            age = year - r["birth_year"]
            if age < 0 or (r.get("death_year") and year > r["death_year"]):
                continue
            for d in diseases:
                if any(c["disease"] == d for c in r["conditions"]):
                    continue
                rate = rate_fn(d, age, r["sex"]) * r.get("risk_mult", {}).get(d, 1.0)   # Phase 0: own multiplier (relatives 1.0)
                
                if any(o is not r and o["degree"] == 1 and 
                       any(c["disease"] == d and c["year"] < year for c in o["conditions"]) 
                       for o in relatives):
                    rate *= cfg["family_multiplier"]
                    
                if rng.random() < min(rate, 0.99):
                    r["conditions"].append({"disease": d, "year": year, "age": age, "confirmed": True})

In [16]:
%%ipytest -q
# Cell 1 - TEST

def test_disease_clusters_in_families():
    rng = np.random.default_rng(7)
    flat = lambda disease, age, sex: 0.01 if age >= 30 else 0.0
    fams = []
    
    for _ in range(1500):
        fam = [{"degree": 1, "sex": "F", "birth_year": 1950, "conditions": []},
               {"degree": 1, "sex": "M", "birth_year": 1950, "conditions": []},
               {"degree": 1, "sex": "F", "birth_year": 1975, "conditions": []}]
        simulate_family_diseases(fam, CFG, rng, flat, diseases=["T2D"])
        fams.append(fam)
        
    has = lambda r: any(c["disease"] == "T2D" for c in r["conditions"])
    with_parent = [f for f in fams if has(f[0]) or has(f[1])]
    without = [f for f in fams if not (has(f[0]) or has(f[1]))]
    
    assert np.mean([has(f[2]) for f in with_parent]) > 1.3 * np.mean([has(f[2]) for f in without])

def test_dead_relative_gets_no_new_disease():
    fam = [{"degree": 1, "sex": "M", "birth_year": 1900, "death_year": 1950, "conditions": []}]
    simulate_family_diseases(fam, CFG, np.random.default_rng(1), lambda d, a, s: 0.5 if a >= 60 else 0.0, ["T2D"])
    assert all(c["year"] <= 1950 for c in fam[0]["conditions"])

.

.

                                                                                           [100%]


2 passed in 0.34s


In [17]:
# Cell 3 - RUN
rng = np.random.default_rng(7)
test_fam = [{"degree": 1, "sex": "F", "birth_year": 1960, "conditions": []},
            {"degree": 1, "sex": "M", "birth_year": 1962, "conditions": []}]
simulate_family_diseases(test_fam, CFG, rng, rate_fn, diseases=["T2D", "CVD"])
print("Simulated Family Conditions:", test_fam)

Simulated Family Conditions: [{'degree': 1, 'sex': 'F', 'birth_year': 1960, 'conditions': []}, {'degree': 1, 'sex': 'M', 'birth_year': 1962, 'conditions': []}]


Part C: Visit schedule

In [18]:
# Cell 2 - CODE
def add_months(d, months):
    m = d.month - 1 + int(round(months))
    return date(d.year + m // 12, m % 12 + 1, 1)

def schedule_visits(rng, cfg, tier_fn=None):
    tier_fn = tier_fn or (lambda d: "Low")
    d = date(cfg["window"]["start_year"] + int(rng.integers(0, 2)), int(rng.integers(1, 13)), 1)
    end = date(cfg["window"]["end_year"], 12, 31)
    dates = []
    
    while d <= end and len(dates) < cfg["visits"]["max"]:
        dates.append(d)
        gap = max(1, cfg["visit_gap_months"][tier_fn(d)] * rng.uniform(0.8, 1.2))
        d = add_months(d, gap)
    return dates

In [19]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def test_visit_dates_go_forward_inside_window():
    dates = schedule_visits(np.random.default_rng(1), CFG)
    assert dates == sorted(dates)
    assert all(date(2021, 1, 1) <= d <= date(2026, 12, 31) for d in dates)
    assert 1 <= len(dates) <= 8

.

                                                                                            [100%]


1 passed in 0.00s


In [20]:
# Cell 3 - RUN
rng = np.random.default_rng(1)
visits = schedule_visits(rng, CFG)
print("Patient Visit Schedule:", visits)

Patient Visit Schedule: [datetime.date(2021, 7, 1), datetime.date(2022, 9, 1), datetime.date(2023, 7, 1), datetime.date(2024, 9, 1), datetime.date(2025, 8, 1), datetime.date(2026, 8, 1)]


Part D: Outcome labels

In [21]:
# Cell 2 - CODE
def label_for(diagnosis_date, index_date, years):
    if diagnosis_date is None:
        return 0
    if diagnosis_date <= index_date:
        return None
    end = date(index_date.year + years, index_date.month, index_date.day)
    return int(diagnosis_date <= end)

In [22]:
%%ipytest -q
# Cell 1 - TEST
from datetime import date

def test_label_is_1_when_diagnosed_inside_window():
    assert label_for(date(2024, 5, 1), date(2021, 3, 1), 5) == 1

def test_label_is_0_when_diagnosed_after_window():
    assert label_for(date(2027, 5, 1), date(2021, 3, 1), 5) == 0

def test_already_ill_at_index_is_excluded():
    assert label_for(date(2020, 1, 1), date(2021, 3, 1), 5) is None

.

.

.

                                                                                          [100%]


3 passed in 0.01s


In [23]:
# Cell 3 - RUN
index = date(2021, 1, 1)
dx = date(2025, 6, 1)
print(f"Diagnosed {dx}, Index {index} -> Label:", label_for(dx, index, 5))

Diagnosed 2025-06-01, Index 2021-01-01 -> Label: 1


Part E: Patients, values, and saving

Values come from the CDSS Data Dictionary (oct31): min/max from the Parameter Ingestion Tables, bands from the Disease Risk Threshold tables. Shares marked AD are author-defined.

**Phase 0 (label-leak fix, Nov 2026):** before a diagnosis, values follow the person's *underlying risk* (age, sex, family history, lifestyle, exposures and random frailty), never their own diagnosis year. The same personal risk multiplies the person's own yearly onset chance, so values and diagnoses share a cause instead of one being read off the other. After a diagnosis values still go High. New settings are in `P0` and `RR` (all author-defined, new, not yet physician-reviewed); sources and reasons: `docs/synthetic_data_changes.md`.

**Physician decisions (Nov 5, 2026):** the CVD label counts hypertension (SBP >= 140 or DBP >= 90 on two visits) as well as a cardiovascular event; BMI uses the Asian cutoffs (overweight >= 23, obese >= 25).

**Phase 0b (Nov 6, 2026):** the training `label` is the true onset within 5 years; `label_recorded` (side analysis) applies the physician-approved rules to the recorded visits (CVD: high BP on two visits or an event; T2D: HbA1c >= 6.5% or FPG >= 126 on two tests; CRD: FEV1/FVC < 70% or a doctor diagnosis of asthma; NEURO: PHQ-9 >= 10 or GAD-7 >= 10 on two visits). Anyone who already meets a rule at the index visit, or whose disease (undiagnosed diabetes included) began by then, is left out of that disease's rows. Only the values that define a label (SBP, DBP, HbA1c, FPG, FEV1/FVC, PHQ-9, GAD-7) and the 2-hour OGTT stay below the rule before the disease starts; BMI and every other value may reach High. BMI is calibrated to FNRI 2021 ENNS (`BMI_CAL`). Acanthosis and allergic rhinitis are calibrated to the physician-approved shares (`P0`).

In [24]:
# Cell 2 - CODE
from pathlib import Path
# ---------------------------------------------------------------------------
# E1. Value table from the CDSS Data Dictionary (oct31)
# min/max = Parameter Ingestion Tables. low/mod/high = draw ranges INSIDE the
# Low / Moderate / High risk bands of the Disease Risk Threshold tables
# (narrowed to typical values: author-defined). kind: req = Required,
# opt = Optional lab, phys = "To confirm with physician" (hospital test).
# drv = which disease pushes the value up: {disease: band when diagnosed}.
# ---------------------------------------------------------------------------
CVD_ = {"CVD": "high"}
HTN_ = {"CVD": "high", "HTN": "high"}
T2D_ = {"T2D": "high"}            # true onset: undiagnosed diabetes raises values too
CRD_ = {"asthma": "high", "COPD": "high"}
COPD_ = {"COPD": "high", "asthma": "mod"}
AST_ = {"asthma": "high"}
DEP_ = {"depression": "high"}
GAD_ = {"GAD": "high"}
NEU_ = {"depression": "high", "GAD": "high"}
NONE_ = {}

def v(mn, mx, low, mod, high, drv, kind, dec=1):
    return {"min": mn, "max": mx, "low": low, "mod": mod, "high": high, "drv": drv, "kind": kind, "dec": dec}

SPEC = {
    # 1. CVD
    "sbp":               v(70, 220, (100, 119), (120, 139), (140, 180), HTN_, "req", 0),
    "dbp":               v(40, 130, (65, 79), (80, 89), (90, 110), HTN_, "req", 0),
    "heart_rate":        v(40, 180, (60, 80), (81, 99), (100, 120), CVD_, "req", 0),
    "total_cholesterol": v(100, 400, (140, 199), (200, 239), (240, 300), CVD_, "opt", 0),
    "hdl":               v(15, 120, (60, 80), (40, 59), (30, 39), CVD_, "opt", 0),
    "ldl":               v(30, 300, (70, 99), (100, 159), (160, 220), CVD_, "opt", 0),
    "triglycerides":     v(40, 1000, (70, 149), (150, 199), (200, 400), T2D_, "opt", 0),
    "hs_crp":            v(0.1, 50, (0.2, 0.9), (1.0, 3.0), (3.1, 10), CVD_, "opt", 1),
    "creatinine":        v(0.4, 10, (0.6, 1.2), (1.3, 1.9), (2.0, 4.0), CVD_, "opt", 2),
    "egfr":              v(5, 140, (90, 120), (60, 89), (30, 59), CVD_, "opt", 0),
    "lvef":              v(15, 75, (55, 70), (41, 52), (25, 40), CVD_, "phys", 0),
    "ctni":              v(0.01, 50, (0.01, 0.03), (0.04, 0.39), (0.40, 2.0), CVD_, "phys", 2),
    "nt_probnp":         v(10, 35000, (20, 124), (125, 449), (450, 3000), CVD_, "phys", 0),
    "sodium_mg":         v(500, 10000, (1500, 2300), (2301, 4000), (4001, 6000), HTN_, "req", 0),
    "activity_min_week": v(0, 1050, (150, 400), (60, 149), (0, 59), CVD_, "req", 0),
    # 2. T2D
    "fpg":               v(50, 450, (75, 99), (100, 125), (126, 250), T2D_, "opt", 0),
    "pg_2h":             v(60, 600, (90, 139), (140, 199), (200, 350), T2D_, "opt", 0),
    "hba1c":             v(4.0, 15.0, (4.6, 5.6), (5.7, 6.4), (6.5, 10.0), T2D_, "opt", 1),
    "insulin":           v(1.0, 100.0, (3.0, 12.0), (12.1, 24.9), (25.0, 50.0), T2D_, "opt", 1),
    "c_peptide":         v(0.1, 12.0, (1.1, 3.0), (3.1, 4.4), (4.5, 8.0), T2D_, "opt", 1),
    "waist_cm":          v(50, 160, (65, 79), (80, 101), (102, 125), T2D_, "req", 0),
    "whr":               v(0.60, 1.40, (0.75, 0.84), (0.85, 0.89), (0.90, 1.05), T2D_, "req", 2),
    "uacr":              v(1, 3500, (5, 29), (30, 299), (300, 1000), T2D_, "opt", 0),
    "uric_acid":         v(1.5, 14.0, (3.5, 5.9), (6.0, 6.9), (7.0, 10.0), T2D_, "opt", 1),
    "carbs_g":           v(50, 600, (130, 250), (251, 350), (351, 500), T2D_, "req", 0),
    "sedentary_hours":   v(1.0, 18.0, (1.0, 3.9), (4.0, 7.9), (8.0, 12.0), T2D_, "req", 1),
    # 3. CRD
    "fev1":              v(0.5, 5.5, (3.0, 4.5), (1.8, 2.9), (0.8, 1.7), CRD_, "opt", 2),
    "fvc":               v(1.0, 7.0, (3.5, 5.5), (2.2, 3.4), (1.2, 2.1), CRD_, "opt", 2),
    "fev1_fvc":          v(25.0, 95.0, (72.0, 85.0), (60.0, 69.9), (40.0, 59.9), COPD_, "opt", 1),
    "fev1_pct_pred":     v(15.0, 120.0, (80.0, 105.0), (50.0, 79.9), (30.0, 49.9), CRD_, "opt", 1),
    "pef":               v(50, 800, (400, 600), (250, 399), (120, 249), CRD_, "opt", 0),
    "eosinophils":       v(0, 2500, (50, 149), (150, 299), (300, 800), AST_, "opt", 0),
    "feno":              v(5, 150, (8, 24), (25, 49), (50, 100), AST_, "opt", 0),
    "spo2":              v(65.0, 100.0, (95.0, 100.0), (90.0, 94.9), (85.0, 89.9), CRD_, "opt", 1),
    "pao2":              v(40, 110, (80, 100), (60, 79), (50, 59), CRD_, "phys", 0),
    "paco2":             v(25, 90, (35, 45), (46, 50), (51, 60), COPD_, "phys", 0),
    "mmrc":              v(0, 4, (0, 1), (2, 2), (3, 4), CRD_, "req", 0),
    "act":               v(5, 25, (20, 25), (16, 19), (8, 15), AST_, "phys", 0),
    "cat":               v(0, 40, (0, 9), (10, 20), (21, 30), COPD_, "phys", 0),
    "ige":               v(2, 3000, (10, 99), (100, 299), (300, 1000), AST_, "opt", 0),
    "exacerbations_year": v(0, 15, (0, 0), (1, 1), (2, 4), CRD_, "phys", 0),
    # 4. NEURO
    "phq9":              v(0, 27, (0, 4), (5, 9), (10, 20), DEP_, "req", 0),
    "gad7":              v(0, 21, (0, 4), (5, 9), (10, 18), GAD_, "req", 0),
    "psqi":              v(0, 21, (2, 5), (6, 10), (11, 16), NEU_, "opt", 0),
    "sleep_hours":       v(1.0, 16.0, (7.0, 9.0), (5.0, 6.9), (3.0, 4.9), NEU_, "req", 1),
    "tsh":               v(0.01, 100.0, (0.5, 4.25), (4.26, 9.99), (10.0, 20.0), NONE_, "opt", 2),
    "vitamin_b12":       v(50, 2000, (250, 900), (100, 199), (60, 99), NONE_, "opt", 0),
}

# Author-defined shares (AD). Change here, then re-run Notebook 2.
AD = {
    "baseline_mod_share": 0.20,      # healthy person, one value already borderline
    "former_smoker": 0.10,           # current smoker share comes from CFG["checks"] (FNRI 20.4%)
    # Phase 0: the next 3 lines are no longer inputs (they made the flag depend on a FUTURE diagnosis).
    # They stay as the physician-approved check values: Notebook 2, Cell 3b reports how close we are.
    "gdm_if_t2d": 0.10, "gdm_other": 0.03,
    "acanthosis_if_t2d": 0.40, "acanthosis_other": 0.05,
    "allergic_rhinitis_if_asthma": 0.60, "allergic_rhinitis_other": 0.15,
    "occupational_dust": 0.15, "alcohol_use": 0.50, "tbi": 0.03,
    "exposure": {"secondhand_smoke": 0.35, "high_salt_diet": 0.20, "processed_diet": 0.20,
                 "low_home_activity": 0.20, "indoor_mold": 0.15, "coal_heating": 0.05},
    "neuro_episode_years": 2,        # PHQ-9/GAD-7 stay high this long after diagnosis
    "neuro_chronic_share": 0.5,      # share with a recurrent/chronic course: PHQ-9/GAD-7 stay high after that
    "height_cm": {"M": (163.5, 6.0), "F": (151.5, 5.5)},
    "htn_scale": 0.6,                # tuned: hypertension without CVD; CVD patients also have high BP
    "t2d_diagnosed_share": 0.48,     # tuned: share of type 2 diabetes that is diagnosed (GBD all vs NOH diagnosed)
}
ALL_DISEASES = ["CVD", "T2D", "asthma", "COPD", "depression", "GAD"]
APP_DOMAINS = {"CVD": ["CVD"], "T2D": ["T2D"], "CRD": ["asthma", "COPD"], "NEURO": ["depression", "GAD"]}

# ---------------------------------------------------------------------------
# E1b. Phase 0 (label-leak fix). Values follow each person's UNDERLYING RISK, never their own
# diagnosis year. Every setting below is author-defined (new, not yet physician-reviewed).
# Sources for each relative risk and the reasons: docs/synthetic_data_changes.md
# ---------------------------------------------------------------------------
P0 = {
    "frailty_sd": 0.5,               # spread of unexplained personal risk (log scale), one draw per disease
    "drift_start_z": 0.0,            # drift starts once the 5-year risk is above the adult average
    "band_per_sd": 0.5,              # average speed: each SD of risk above average moves a value half a band
    "speed_person_sd": 0.5,          # how much drift speed differs between people (log-normal)
    "speed_value_sd": 0.25,          # how much one person's values differ from each other (log-normal)
    "lifestyle_values": ["sodium_mg", "activity_min_week", "carbs_g", "sedentary_hours"],  # risk inputs, no drift
    "gdm_share_female": 0.04,        # history of gestational diabetes; = overall share of women in the previous run
    "acanthosis_or_per_sd": 2.0,     # acanthosis odds x2 for each SD of type 2 diabetes risk (a marker, not a cause)
    # Phase 0b: CALIBRATED to the physician-approved shares (Notebook 2, Cell 3b), the way prevalence_factor is
    # calibrated to GBD. Only people whose disease began BEFORE the window get the "already ill" share; all of
    # them are left out of that disease's training rows, so a flag still never tells the model a future diagnosis.
    "allergic_rhinitis_share": 0.175,            # before any disease; published 10-32% of adults (Pawankar 2012)
    "allergic_rhinitis_added_if_asthma_before_window": 0.46,  # rhinitis 60-80% of asthma (Pawankar 2012)
    "acanthosis_share_at_avg_risk": 0.028,        # people without type 2 diabetes yet, at average risk
    "acanthosis_share_if_t2d_before_window": 0.49,  # published 25-67% of adults with T2D (Bahadursingh 2014)
}
# Relative risk on the person's OWN yearly onset chance (relatives are simulated as before).
# HR / OR used as RR; factors multiply (assumed independent). Sources: docs/synthetic_data_changes.md
RR = {
    "CVD":        {"smoker_current": 1.63, "secondhand_smoke": 1.25, "high_salt_diet": 1.14, "low_activity": 1.16},
    "HTN":        {},
    "T2D":        {"smoker_current": 1.37, "smoker_former": 1.14, "processed_diet": 1.19,
                   "low_home_activity": 1.35, "sedentary": 1.20, "gestational_diabetes": 9.51},
    "asthma":     {"allergic_rhinitis": 3.53, "indoor_mold": 1.29, "secondhand_smoke": 1.97},
    "COPD":       {"smoker_current": 3.51, "smoker_former": 2.35, "occupational_dust": 1.22,
                   "coal_heating": 2.80, "secondhand_smoke": 1.55},
    "depression": {"tbi_history": 1.59},
    "GAD":        {},
}
RISK_DISEASES = ALL_DISEASES + ["HTN"]
BMI_BANDS = {"low": (18.5, 22.9), "mod": (23.0, 24.9), "high": (25.0, 40.0)}   # Asian cutoffs (physician, Nov 5)
# BMI calibrated to FNRI 2021 ENNS (DOST-FNRI 2022, National Results Dissemination), adults 20-59 and 60+
# (decision Nov 6), the way prevalence_factor is calibrated to GBD. Author-defined (new, not yet physician-reviewed);
# targets and source: docs/synthetic_data_changes.md. BMI replaces baseline_mod_share with its own starting bands;
# the Low band has an underweight part (16.0-18.4) and the High band a part at 30 or more. Age group = age at the
# window start (fixed per person, so BMI does not jump at 60).
BMI_CAL = {
    "20-59": {"start_mod": 0.03,              # share starting in 23-24.9 (before any drift)
              "start_high": 0.322,            # share starting at 25 or more
              "underweight_of_low": 0.242,    # part of the Low band's positions below 18.5
              "obese30_of_high": 0.24},       # part of the High band's positions at 30 or more
    "60+":   {"start_mod": 0.03,              # = 20-59: ENNS 60+ gives no 23.0 split (WHO cut-offs only)
              "start_high": 0.169, "underweight_of_low": 0.333, "obese30_of_high": 0.21},
}
BMI_OLDER_FROM = 60
BMI_UNDERWEIGHT = (16.0, 18.4)
BMI_HIGH_PARTS = ((25.0, 29.9), (30.0, 40.0))

# ---------------------------------------------------------------------------
# E1c. Physician-approved label rules (signed Nov 5, 2026), applied to the recorded visits:
#   CVD   SBP >= 140 or DBP >= 90 on two visits, or a cardiovascular event
#   T2D   HbA1c >= 6.5% or FPG >= 126 mg/dL on two tests (HbA1c and FPG from one blood sample = two tests, ADA)
#   CRD   FEV1/FVC below 70%, or a doctor diagnosis of asthma
#   NEURO PHQ-9 >= 10 or GAD-7 >= 10 on two visits
# The training LABEL is the generator's true onset within 5 years (decision Nov 6): from onset on, the values
# meet the rule whenever they are measured. LABEL_RECORDED applies the rule to the recorded visits only (side
# analysis). Both use the same exclusion: the disease began, or the index visit already meets the rule.
# A value that defines a label stays below the rule's cut-off until the disease starts (Phase 0b: only these and
# the 2-hour OGTT, which is held below 200 mg/dL like HbA1c and FPG; every other value, BMI included, may reach
# High before diagnosis). FEV1/FVC stays in Low: its Moderate band (60-69.9%) is already below 70%.
# ---------------------------------------------------------------------------
RULE_TIMES = {"CVD": 2, "T2D": 2, "CRD": 1, "NEURO": 2}          # abnormal visits (T2D: tests) needed
RULE_ROUTE = {"CVD": ["CVD"], "T2D": [], "CRD": ["asthma"], "NEURO": []}   # a diagnosis that counts on its own
PRE_DX_CAP = {"sbp": 1.999, "dbp": 1.999, "hba1c": 1.999, "fpg": 1.999, "pg_2h": 1.999, "phq9": 1.999,
              "gad7": 1.999, "fev1_fvc": 0.999}                                # highest drift position before the disease starts
# Decision Nov 6: a value that is a CAUSE of a disease (risk factor or risk marker), not a consequence, is never
# forced to High by that diagnosis; it keeps following the person's own path (drift with risk; lifestyle values:
# their baseline). Classification and reasons: docs/synthetic_data_changes.md.
NOT_FORCED_AFTER_DX = {"bmi", "waist_cm", "whr",                                 # adiposity -> type 2 diabetes
                       "total_cholesterol", "ldl", "hdl",                        # lipids -> CVD
                       "sodium_mg", "activity_min_week", "carbs_g", "sedentary_hours"}   # lifestyle
OTHER_CAP = float("inf")      # any other value: no cap; past the start of High it nears the top of the band (band_and_u)

def _num(v, k):
    x = v.get(k)
    return None if x is None or pd.isna(x) else x

def abnormal_tests(domain, v):
    """How many of the rule's tests are abnormal at one visit (a test not done is not abnormal)."""
    ge = lambda k, cut: _num(v, k) is not None and _num(v, k) >= cut
    if domain == "CVD":
        return int(ge("sbp", 140) or ge("dbp", 90))
    if domain == "T2D":
        return int(ge("hba1c", 6.5)) + int(ge("fpg", 126))
    if domain == "CRD":
        return int(_num(v, "fev1_fvc") is not None and v["fev1_fvc"] < 70)
    if domain == "NEURO":
        return int(ge("phq9", 10) or ge("gad7", 10))
    raise KeyError(domain)

def dx_date(year):
    return None if year is None else date(int(year), 7, 1)   # diagnosis = middle of that year

# ---------------------------------------------------------------------------
# E2. Ages and sexes from PSA 2024 projected population (adults 20-79)
# ---------------------------------------------------------------------------
def load_age_weights(pop_path, year="2024"):
    p = pd.read_csv(pop_path, skiprows=1)
    p = p[p["Sex"].isin(["Male", "Female"]) & p["Age Group"].str.match(r"^[2-7]\d-")]
    p = p.assign(lo=p["Age Group"].str.split("-").str[0].astype(int), w=p[year])
    return p[["Sex", "lo", "w"]].reset_index(drop=True)

AGE_W = load_age_weights(CFG["pop_file"], CFG["death_year"])

def hypertension_chance(age):
    # rises with age; tuned so adults average about 20% (FNRI target in CFG["checks"]) - shape AD
    return AD["htn_scale"] * float(np.clip(0.02 + 0.007 * (age - 20), 0.02, 0.45))

HTN_START_AGE = 20

def htn_hazard(age):
    """Yearly chance of NEW hypertension at this age, from the approved prevalence curve above
    (the rise in prevalence from one year to the next, among those without it). At age 20 = the prevalence."""
    if age < HTN_START_AGE:
        return 0.0
    if age == HTN_START_AGE:
        return hypertension_chance(age)
    before = hypertension_chance(age - 1)
    return max(0.0, hypertension_chance(age) - before) / (1 - before)

# ---------------------------------------------------------------------------
# E2b. Personal risk (Phase 0): risk factors -> personal multiplier -> 5-year risk -> z
# ---------------------------------------------------------------------------
def factor_shares(sex):
    """Population share of each risk factor (the same numbers that draw them), for the normaliser."""
    shares = {"smoker_current": CFG["checks"]["current_smoker"], "smoker_former": AD["former_smoker"],
              **AD["exposure"], "low_activity": AD["baseline_mod_share"], "sedentary": AD["baseline_mod_share"],
              "gestational_diabetes": P0["gdm_share_female"] if sex == "F" else 0.0,
              "allergic_rhinitis": P0["allergic_rhinitis_share"], "occupational_dust": AD["occupational_dust"],
              "tbi_history": AD["tbi"]}
    return shares

def mean_raw_multiplier(disease, sex):
    """Average of exp(sum log RR x) + frailty over the population, so the final multiplier averages 1
    and the GBD-calibrated rates (and prevalence_factor) keep their meaning. Computed, not a setting."""
    rr, sh = RR[disease], factor_shares(sex)
    pc, pf = sh["smoker_current"], sh["smoker_former"]
    total = (1 - pc - pf) + pc * rr.get("smoker_current", 1.0) + pf * rr.get("smoker_former", 1.0)
    for k, r in rr.items():
        if k not in ("smoker_current", "smoker_former"):
            total *= 1 - sh[k] + sh[k] * r
    return total * float(np.exp(P0["frailty_sd"] ** 2 / 2))

def risk_factors(p):
    """0/1 risk factors of one patient (all drawn before the patient's own diseases)."""
    return {"smoker_current": int(p["smoking_status"] == "current"),
            "smoker_former": int(p["smoking_status"] == "former"),
            **{k: int(x) for k, x in p["exposures"].items()},
            "low_activity": int(p["base_mod"]["activity_min_week"]),    # below 150 min/week before any diagnosis
            "sedentary": int(p["base_mod"]["sedentary_hours"]),         # 4 h/day or more before any diagnosis
            "gestational_diabetes": p["gestational_diabetes"], "allergic_rhinitis": p["allergic_rhinitis"],
            "occupational_dust": p["occupational_dust"], "tbi_history": p["tbi_history"]}

def personal_multipliers(factors, sex, rng):
    """One multiplier per disease: published relative risks x random frailty, divided by the population mean."""
    out = {}
    for d in RISK_DISEASES:
        log_m = sum(np.log(r) * factors[k] for k, r in RR[d].items()) + P0["frailty_sd"] * rng.normal()
        out[d] = float(np.exp(log_m)) / mean_raw_multiplier(d, sex)
    return out

def yearly_hazard(disease, age, sex):
    return htn_hazard(age) if disease == "HTN" else rate_fn(disease, age, sex)

def five_year_risk(disease, age, sex, multiplier=1.0):
    free = 1.0
    for k in range(5):
        free *= 1 - min(yearly_hazard(disease, age + k, sex) * multiplier, 0.99)
    return max(1 - free, 1e-9)

def reference_log_risk(disease):
    """Mean and SD of log 5-year risk in the adult population (PSA age/sex weights, no family history,
    multiplier 1). Computed once from the rates; it never looks at anyone's outcome."""
    vals, w = [], []
    for row in AGE_W.itertuples(index=False):
        sex = "M" if row.Sex == "Male" else "F"
        for age in range(int(row.lo), int(row.lo) + 5):
            vals.append(np.log(five_year_risk(disease, age, sex)))
            w.append(row.w / 5)
    vals, w = np.array(vals), np.array(w) / np.sum(w)
    mean = float(np.sum(w * vals))
    return mean, float(np.sqrt(np.sum(w * (vals - mean) ** 2)))

REF_LOG_RISK = {d: reference_log_risk(d) for d in RISK_DISEASES}

def family_multiplier_now(p, disease, year):
    """x family_multiplier once a first-degree relative had the disease (the same rule as Part B)."""
    if disease == "HTN":
        return 1.0
    hit = any(r["degree"] == 1 and any(c["disease"] == disease and c["year"] < year for c in r["conditions"])
              for r in p["relatives"])
    return CFG["family_multiplier"] if hit else 1.0

def risk_z(p, d):
    """Each disease's 5-year risk on date d as SDs above the adult average. Uses age, sex, relatives
    diagnosed so far and the personal multiplier. Never uses the patient's own diagnosis."""
    age = d.year - p["birth_year"]
    out = {}
    for dis in RISK_DISEASES:
        m = p["risk_mult"][dis] * family_multiplier_now(p, dis, d.year)
        mean, sd = REF_LOG_RISK[dis]
        out[dis] = (np.log(five_year_risk(dis, age, p["sex"], m)) - mean) / sd
    return out

# ---------------------------------------------------------------------------
# E3. make_patient: one patient + family tree + diseases over time
# ---------------------------------------------------------------------------
def _relative(rel, degree, sex, birth_year, rng):
    return {"relation": rel, "degree": degree, "sex": sex, "birth_year": int(birth_year),
            "death_year": draw_death_year(int(birth_year), sex, rng, DR), "conditions": []}

def make_family(birth_year, rng):
    fam = []
    for rel, sex in (("father", "M"), ("mother", "F")):
        by = birth_year - int(rng.integers(20, 41))
        fam.append(_relative(rel, 1, sex, by, rng))
        for gsex in ("M", "F"):
            fam.append(_relative(f"{rel}'s {'father' if gsex == 'M' else 'mother'}", 2, gsex,
                                 by - int(rng.integers(20, 41)), rng))
    for _ in range(min(8, rng.poisson(mean_siblings(birth_year)))):
        fam.append(_relative("sibling", 1, rng.choice(["M", "F"]),
                             birth_year + int(rng.integers(-10, 11)), rng))
    return fam

def known_family(relatives):
    """What the patient can report: the family without undiagnosed diabetes (used for FHRS)."""
    return [{**r, "conditions": [c for c in r["conditions"] if c.get("diagnosed", True)]} for r in relatives]

def make_patient(pid, rng, cfg=CFG):
    row = AGE_W.iloc[rng.choice(len(AGE_W), p=AGE_W["w"] / AGE_W["w"].sum())]
    sex = "M" if row["Sex"] == "Male" else "F"
    age0 = int(row["lo"]) + int(rng.integers(0, 5))           # age on 1 Jan of the window start
    birth_year = cfg["window"]["start_year"] - age0

    female = sex == "F"
    mu, sd = AD["height_cm"][sex]

    # 1. Who the person is BEFORE any of their own diseases: lifestyle, exposures, history (Phase 0 order)
    smoke_p = cfg["checks"]["current_smoker"]
    s = rng.random()
    status = "current" if s < smoke_p else "former" if s < smoke_p + AD["former_smoker"] else "never"
    ppd = float(rng.uniform(0.25, 1.5))
    years = max(0, age0 - 18) if status == "current" else float(rng.uniform(2, 20)) if status == "former" else 0
    p = {
        "patient_id": pid, "sex": sex, "birth_year": birth_year,
        "height_cm": round(float(np.clip(rng.normal(mu, sd), 135, 195)), 1),
        "u": {k: float(rng.random()) for k in list(SPEC) + ["bmi"]},          # personal position in a band
        "base_mod": {k: bool(rng.random() < AD["baseline_mod_share"]) for k in list(SPEC) + ["bmi"]},
        "smoking_status": status, "packs_per_day": ppd if status != "never" else 0.0,
        "pack_years_start": min(120.0, ppd * years) if status != "never" else 0.0,
        "exposures": {k: bool(rng.random() < q) for k, q in AD["exposure"].items()},
        "gestational_diabetes": int(female and rng.random() < P0["gdm_share_female"]),
        "allergic_rhinitis": int(rng.random() < P0["allergic_rhinitis_share"]),
        "occupational_dust": int(rng.random() < AD["occupational_dust"]),
        "alcohol_use": int(rng.random() < AD["alcohol_use"]),
        "tbi_history": int(rng.random() < AD["tbi"]),
        "neuro_chronic": bool(rng.random() < AD["neuro_chronic_share"]),
        "t2d_diagnosed": bool(rng.random() < AD["t2d_diagnosed_share"]),
        "speed": float(np.exp(P0["speed_person_sd"] * rng.normal())),
        "speed_value": {k: float(np.exp(P0["speed_value_sd"] * rng.normal())) for k in list(SPEC) + ["bmi"]},
    }
    p["risk_mult"] = personal_multipliers(risk_factors(p), sex, rng)

    # 2. The person's own diseases: same rates and family rule as before, x their personal multiplier
    me = {"relation": "self", "degree": 0, "sex": sex, "birth_year": birth_year,
          "death_year": None, "conditions": [], "risk_mult": p["risk_mult"]}
    family = make_family(birth_year, rng)
    simulate_family_diseases(family + [me], cfg, rng, rate_fn, ALL_DISEASES,
                             end_year=cfg["window"]["end_year"] + 1)
    for r in family:                                           # what the patient knows
        for c in r["conditions"]:
            c["confirmed"] = bool(rng.random() < cfg["relatives"]["confirmed_share"])
            if rng.random() < cfg["relatives"]["unknown_onset_share"]:
                c["age"] = None
            if c["disease"] == "T2D":                          # undiagnosed diabetes: not known to the family
                c["diagnosed"] = bool(rng.random() < AD["t2d_diagnosed_share"])

    dx = {c["disease"]: c["year"] for c in me["conditions"]}
    dx = {d: dx.get(d) for d in ALL_DISEASES}
    dx["HTN"] = None                                           # hypertension: an onset year (was: always / never)
    for year in range(birth_year + HTN_START_AGE, cfg["window"]["end_year"] + 2):
        if rng.random() < min(htn_hazard(year - birth_year) * p["risk_mult"]["HTN"], 0.99):
            dx["HTN"] = year
            break
    p.update({"relatives": family, "dx": dx, "known_relatives": known_family(family),
              "childhood_asthma": int(dx["asthma"] is not None and dx["asthma"] - birth_year < 18)})

    # 3. Acanthosis: a marker of type 2 diabetes RISK at the window start, or of diabetes that began before it
    p["acanthosis_nigricans"] = int(rng.random() < acanthosis_chance(p, cfg))
    # 4. Allergic rhinitis also comes with asthma (united airway): people whose asthma began before the window
    p["allergic_rhinitis_before_disease"] = p["allergic_rhinitis"]   # the draw that set the asthma risk
    if began_before_window(p, "asthma", cfg) and not p["allergic_rhinitis"]:
        p["allergic_rhinitis"] = int(rng.random() < P0["allergic_rhinitis_added_if_asthma_before_window"])
    # 5. BMI starting band (calibrated to ENNS 2021; see BMI_CAL)
    p["bmi_start"] = float(rng.random())
    p["bmi_group"] = "60+" if age0 >= BMI_OLDER_FROM else "20-59"
    return p

def bmi_start_band(p):
    cal, r = BMI_CAL[p["bmi_group"]], p["bmi_start"]
    return 2 if r < cal["start_high"] else 1 if r < cal["start_high"] + cal["start_mod"] else 0

def bmi_value(level, frac, cal):
    """BMI from band and position (0 = lowest BMI). Low: underweight part, then 18.5-22.9; High: 25-29.9,
    then 30-40. Moderate: 23-24.9. cal = BMI_CAL[age group]."""
    def lin(a, b, f):
        return a + (b - a) * min(max(f, 0.0), 1.0)
    if level == "low":
        uw = cal["underweight_of_low"]
        return lin(*BMI_UNDERWEIGHT, frac / uw) if frac < uw else lin(*BMI_BANDS["low"], (frac - uw) / (1 - uw))
    if level == "high":
        cut = 1 - cal["obese30_of_high"]
        return lin(*BMI_HIGH_PARTS[0], frac / cut) if frac < cut else lin(*BMI_HIGH_PARTS[1], (frac - cut) / (1 - cut))
    return lin(*BMI_BANDS["mod"], frac)

def began_before_window(p, disease, cfg=CFG):
    """True onset before 1 Jan of the window start: such people are left out of that disease's rows."""
    when = dx_date(p["dx"].get(disease))
    return when is not None and when < date(cfg["window"]["start_year"], 1, 1)

def acanthosis_chance(p, cfg=CFG):
    """Diabetes (diagnosed or not) that began before the window: the calibrated published share.
    Anyone else: logistic in their type 2 diabetes risk at the window start. Never a future diagnosis."""
    if began_before_window(p, "T2D", cfg):
        return P0["acanthosis_share_if_t2d_before_window"]
    z = risk_z(p, date(cfg["window"]["start_year"], 1, 1))["T2D"]
    base = P0["acanthosis_share_at_avg_risk"]
    logit = np.log(base / (1 - base)) + np.log(P0["acanthosis_or_per_sd"]) * z
    return float(1 / (1 + np.exp(-logit)))

# ---------------------------------------------------------------------------
# E4. make_visit_values: one visit's measurements
# ---------------------------------------------------------------------------
LEVELS = ["low", "mod", "high"]

def level_for(p, name, d):
    """Band forced by a diagnosis ON OR BEFORE date d (None = no diagnosis yet): high after diagnosis
    (neuro: mod once an episode is over). Phase 0: a FUTURE diagnosis is never looked at.
    A cause of the disease is never forced (NOT_FORCED_AFTER_DX, decision Nov 6)."""
    if name in NOT_FORCED_AFTER_DX:
        return None
    best = None
    for disease, lvl in SPEC_DRV(name).items():
        when = dx_date(p["dx"].get(disease))
        if when is None or when > d:
            continue
        if (disease in ("depression", "GAD") and d.year - when.year > AD["neuro_episode_years"]
                and not p.get("neuro_chronic", False)):
            lvl = "mod"                                         # episode over, still not fully well
        best = lvl if best is None else max(best, lvl, key=LEVELS.index)
    return best

def drift_position(p, name, d, z):
    """Where a value sits before diagnosis: 0-1 = Low band, 1-2 = Moderate, 2-3 = High (the position inside
    the band is the fraction, 0 = healthiest end). Starts at the personal baseline and moves up as the
    person's risk (z) builds, at the person's own speed. Lifestyle values do not drift: they are risk inputs.
    Phase 0b: only a value that defines a label is capped below its rule (PRE_DX_CAP)."""
    cap = PRE_DX_CAP.get(name, OTHER_CAP)
    band0 = bmi_start_band(p) if name == "bmi" else (1 if p["base_mod"][name] else 0)
    start = min(p["u"][name] + band0, cap)
    drivers = list(SPEC_DRV(name))
    if name in P0["lifestyle_values"] or not drivers:
        return start
    excess = max(0.0, max(z[dis] for dis in drivers) - P0["drift_start_z"])
    move = p["speed"] * p["speed_value"][name] * P0["band_per_sd"] * excess
    return min(start + move, cap)

def band_and_u(p, name, d, z):
    """Band and position in it: the higher of the drift position and the diagnosis band."""
    pos = drift_position(p, name, d, z)
    i = min(int(pos), 2)
    lvl = level_for(p, name, d)
    if lvl is not None and LEVELS.index(lvl) > i:
        return lvl, p["u"][name]
    # High: the position nears the top of the band gradually (no pile-up at the band maximum)
    return LEVELS[i], (1 - np.exp(-(pos - 2))) if i == 2 else pos - i

def SPEC_DRV(name):
    return T2D_ if name == "bmi" else SPEC[name]["drv"]

def ckd_epi_2021(creatinine, age, sex):
    """eGFR (mL/min/1.73m2), CKD-EPI 2021 race-free equation. sex: 'M'/'F' or 'Male'/'Female'."""
    female = sex in ("F", "Female")
    k, a = (0.7, -0.241) if female else (0.9, -0.302)
    s = creatinine / k
    return 142 * min(s, 1) ** a * max(s, 1) ** -1.2 * 0.9938 ** age * (1.012 if female else 1)

def draw(band, u):
    a, b = band
    return a + (b - a) * u

def lower_is_worse(name):
    """HDL, SpO2, FEV1/FVC, sleep hours ...: the Moderate band lies below the Low band."""
    s = SPEC[name]
    return s["mod"][0] < s["low"][0]

def make_visit_values(p, d, rng, cfg=CFG):
    age = d.year - p["birth_year"]
    labs_done = rng.random() >= cfg["missing"]["optional_lab"]
    hospital_done = rng.random() >= cfg["missing"]["hospital_only"]
    out = {"age": age}
    z = risk_z(p, d)
    for name, s in SPEC.items():
        if (s["kind"] == "opt" and not labs_done) or (s["kind"] == "phys" and not hospital_done):
            out[name] = np.nan
            continue
        lvl, u0 = band_and_u(p, name, d, z)
        u = float(np.clip(u0 + rng.normal(0, 0.05), 0, 1))
        val = draw(s[lvl], 1 - u if lower_is_worse(name) else u)      # Phase 0b: higher position = worse value
        val = float(np.clip(round(val, s["dec"]), s["min"], s["max"]))
        out[name] = int(val) if s["dec"] == 0 else val
    # FEV1 from FVC and the drawn ratio, then the ratio from the rounded values (computed field)
    if not np.isnan(out["fvc"]):
        out["fev1"] = float(np.clip(round(out["fvc"] * out["fev1_fvc"] / 100, 2), 0.5, 5.5))
        out["fev1_fvc"] = round(out["fev1"] / out["fvc"] * 100, 1)
    # eGFR from creatinine, age and sex: CKD-EPI 2021 (computed field)
    if not np.isnan(out["creatinine"]):
        out["egfr"] = int(np.clip(round(ckd_epi_2021(out["creatinine"], age, p["sex"])), 5, 140))
    # BMI from weight and height (dictionary: computed)
    bmi_lvl, bmi_u = band_and_u(p, "bmi", d, z)
    h = p["height_cm"] / 100
    target = bmi_value(bmi_lvl, float(np.clip(bmi_u + rng.normal(0, 0.03), 0, 1)), BMI_CAL[p["bmi_group"]])
    out["height_cm"] = p["height_cm"]
    out["weight_kg"] = round(target * h * h, 1)
    out["bmi"] = round(out["weight_kg"] / h ** 2, 1)
    # HOMA-IR from FPG and insulin (dictionary: computed)
    out["homa_ir"] = (round(float(np.clip(out["fpg"] * out["insulin"] / 405, 0.2, 25.0)), 1)
                      if not (np.isnan(out["fpg"]) or np.isnan(out["insulin"])) else np.nan)
    # smoking: pack-years only go up for current smokers
    years_since_start = d.year - cfg["window"]["start_year"]
    extra = p["packs_per_day"] * years_since_start if p["smoking_status"] == "current" else 0.0
    out["pack_years"] = round(min(120.0, p["pack_years_start"] + extra), 1)
    out["smoking_status"] = p["smoking_status"]
    for k in ("gestational_diabetes", "acanthosis_nigricans", "allergic_rhinitis", "childhood_asthma",
              "occupational_dust", "alcohol_use", "tbi_history"):
        out[k] = p[k]
    return out

# ---------------------------------------------------------------------------
# E5. Training rows (index date = first visit) and the generator
# ---------------------------------------------------------------------------
def first_dx(p, domain):
    years = [p["dx"][d] for d in APP_DOMAINS[domain] if p["dx"].get(d) is not None]
    return dx_date(min(years)) if years else None

def high_bp(sbp, dbp):
    return sbp >= 140 or dbp >= 90

def already_ill(p, domain, index_date, index_values=None):
    """Left out of the domain's rows: the disease truly began on or before the index date (undiagnosed
    diabetes and, for CVD, hypertension included), or the index visit already meets the rule's cut-off."""
    onsets = [first_dx(p, domain)] + ([dx_date(p["dx"].get("HTN"))] if domain == "CVD" else [])
    if any(w is not None and w <= index_date for w in onsets):
        return True
    return index_values is not None and abnormal_tests(domain, index_values) > 0

def outcome_date(p, domain, index_date, visits=()):
    """First date after the index date on which the approved rule is met: the diagnosis the rule names
    (CVD event, asthma) or the visit that brings the abnormal count to RULE_TIMES.
    visits: [(visit_date, values), ...] in date order."""
    dates = [dx_date(p["dx"].get(d)) for d in RULE_ROUTE[domain]]
    count = 0
    for d, v in visits or ():
        if d <= index_date:
            continue
        count += abnormal_tests(domain, v)
        if count >= RULE_TIMES[domain]:
            dates.append(d)
            break
    dates = [x for x in dates if x is not None and x > index_date]
    return min(dates) if dates else None

def true_onset(p, domain):
    """The generator's onset: the first disease of the domain (CVD: an event or hypertension)."""
    dates = [first_dx(p, domain)] + ([dx_date(p["dx"].get("HTN"))] if domain == "CVD" else [])
    dates = [x for x in dates if x is not None]
    return min(dates) if dates else None

def training_rows(p, index_date, cfg=CFG, visits=None):
    """label = true onset within 5 years (decision Nov 6); label_recorded = the approved rule on the recorded
    visits (side analysis). Already ill at index (by onset or by the index visit) -> no row."""
    index_values = next((v for d, v in visits or () if d == index_date), None)
    years = cfg["label_window_years"]
    rows = []
    for domain in APP_DOMAINS:
        if already_ill(p, domain, index_date, index_values):
            continue
        rows.append({"patient_id": p["patient_id"], "domain": domain, "index_date": index_date,
                     "label": label_for(true_onset(p, domain), index_date, years),
                     "label_recorded": label_for(outcome_date(p, domain, index_date, visits), index_date, years)})
    return rows

def has_flags(p, d, high_bp_visits=None, first_visit=None):
    """has_<domain> = diagnosed by date d (what the app knows); has_T2D_any also counts undiagnosed diabetes.
    has_CVD = a cardiovascular event (has_CV_event) OR hypertension (has_HTN): known before the first
    visit, or high BP on two visits so far. Without visit counts (scheduling): the true hypertension state."""
    out = {}
    for dom in DOMAIN:
        w = first_dx(p, dom)
        ill = int(w is not None and w <= d)
        out[f"has_{dom}"] = ill * (int(p["t2d_diagnosed"]) if dom == "T2D" else 1)
        if dom == "T2D":
            out["has_T2D_any"] = ill
    htn = dx_date(p["dx"].get("HTN"))
    if high_bp_visits is None:
        has_htn = htn is not None and htn <= d
    else:
        has_htn = (htn is not None and first_visit is not None and htn <= first_visit) or high_bp_visits >= 2
    out["has_CV_event"] = out["has_CVD"]
    out["has_HTN"] = int(has_htn)
    out["has_CVD"] = int(out["has_CV_event"] or has_htn)
    return out

def tier_for(p, d):
    if any(v for k, v in has_flags(p, d).items() if k != "has_T2D_any"):
        return "High"
    if max(band(fhrs(p["known_relatives"], p["exposures"], dom, d, CFG)) for dom in DOMAIN) >= 3:
        return "Moderate"
    return "Low"

def generate(cfg=CFG, n=None, out_dir="output"):
    n = cfg["n_patients"] if n is None else n
    rows_p, rows_r, rows_v, rows_t = [], [], [], []
    for pid in range(n):
        rng = np.random.default_rng([cfg["seed"], pid])
        p = make_patient(pid, rng, cfg)
        visits = schedule_visits(rng, cfg, tier_fn=lambda d: tier_for(p, d))
        recorded, n_high = [], 0
        for i, d in enumerate(visits):
            vals = make_visit_values(p, d, rng, cfg)
            for dom in DOMAIN:
                score = fhrs(p["known_relatives"], p["exposures"], dom, d, cfg)
                vals[f"fhrs_{dom}"], vals[f"fhrs_band_{dom}"] = score, band(score)
            recorded.append((d, vals))
            n_high += high_bp(vals["sbp"], vals["dbp"])
            vals.update(has_flags(p, d, high_bp_visits=n_high, first_visit=visits[0]))
            rows_v.append({"patient_id": pid, "visit_no": i + 1, "visit_date": d, **vals})
        rows_t += training_rows(p, visits[0], cfg, visits=recorded)
        rows_p.append({"patient_id": pid, "sex": "Male" if p["sex"] == "M" else "Female",
                       "birth_year": p["birth_year"], "height_cm": p["height_cm"],
                       "smoking_status": p["smoking_status"], "t2d_diagnosed": int(p["t2d_diagnosed"]),
                       **{k: p[k] for k in ("gestational_diabetes", "acanthosis_nigricans", "allergic_rhinitis",
                                            "childhood_asthma", "occupational_dust", "alcohol_use", "tbi_history")},
                       **{f"exp_{k}": int(x) for k, x in p["exposures"].items()},
                       **{f"true_dx_year_{k}": y for k, y in p["dx"].items()},
                       "is_synthetic": True})
        for j, r in enumerate(p["relatives"]):
            base = {"patient_id": pid, "relative_no": j + 1, "relation": r["relation"], "degree": r["degree"],
                    "sex": r["sex"], "birth_year": r["birth_year"], "death_year": r["death_year"]}
            conds = r["conditions"] or [{"disease": None, "year": None, "age": None, "confirmed": None, "diagnosed": None}]
            for c in conds:
                rows_r.append({**base, "disease": c["disease"], "dx_year": c["year"], "dx_age": c["age"],
                               "confirmed": c["confirmed"], "diagnosed": c.get("diagnosed", True)})
    out = {"patients": pd.DataFrame(rows_p), "relatives": pd.DataFrame(rows_r),
           "visits": pd.DataFrame(rows_v), "training": pd.DataFrame(rows_t)}
    if out_dir is not None:
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        for name, df in out.items():
            df.to_csv(Path(out_dir) / f"{name}.csv", index=False)
    return out

In [25]:
%%ipytest -q
# Cell 1 - TEST
import copy
import pytest

def small_cfg(**over):
    c = copy.deepcopy(CFG)
    c.update(over)
    return c

@pytest.fixture(scope="module")
def sample():
    return generate(CFG, n=150, out_dir=None)

def test_every_value_stays_inside_its_dictionary_min_max_range(sample):
    v = sample["visits"]
    for name, s in SPEC.items():
        col = v[name].dropna()
        assert col.between(s["min"], s["max"]).all(), name
    assert v["bmi"].between(12, 60).all() and v["homa_ir"].dropna().between(0.2, 25).all()

def test_bmi_equals_weight_divided_by_height_squared_rounded_to_0_1(sample):
    v = sample["visits"]
    assert (v["bmi"] == (v["weight_kg"] / (v["height_cm"] / 100) ** 2).round(1)).all()

def test_a_progressors_hba1c_is_higher_at_the_last_visit_than_at_the_first():
    cfg = small_cfg(missing={"optional_lab": 0.0, "hospital_only": 0.0})
    rng = np.random.default_rng(5)
    p = make_patient(1, rng, cfg)
    p["dx"]["T2D"] = 2024                       # becomes diabetic inside the window
    first = make_visit_values(p, date(2021, 3, 1), rng, cfg)
    last = make_visit_values(p, date(2026, 3, 1), rng, cfg)
    assert last["hba1c"] > first["hba1c"]
    assert last["hba1c"] >= 6.5 > first["hba1c"]

def test_gestational_diabetes_appears_only_for_female_patients(sample):
    pts = sample["patients"]
    assert (pts.loc[pts["sex"] == "Male", "gestational_diabetes"] == 0).all()
    assert pts.loc[pts["sex"] == "Female", "gestational_diabetes"].sum() > 0

def test_pack_years_never_go_down_between_visits(sample):
    v = sample["visits"].sort_values(["patient_id", "visit_date"])
    assert (v.groupby("patient_id")["pack_years"].diff().dropna() >= 0).all()

def test_required_values_are_never_missing(sample):
    v = sample["visits"]
    for name, s in SPEC.items():
        if s["kind"] == "req":
            assert v[name].notna().all(), name

def test_age_is_18_to_100_at_every_visit(sample):
    assert sample["visits"]["age"].between(18, 100).all()

def test_patient_already_ill_at_index_has_no_training_row():
    rng = np.random.default_rng(9)
    p = make_patient(2, rng, CFG)
    p["dx"]["CVD"] = 2000
    rows = training_rows(p, date(2021, 5, 1), CFG)
    assert "CVD" not in [r["domain"] for r in rows]

def test_generator_writes_four_linked_files(tmp_path):
    generate(CFG, n=20, out_dir=tmp_path)
    files = {f: pd.read_csv(tmp_path / f) for f in ["patients.csv", "relatives.csv", "visits.csv", "training.csv"]}
    ids = set(files["patients.csv"]["patient_id"])
    assert len(ids) == 20
    for f in ["relatives.csv", "visits.csv", "training.csv"]:
        assert set(files[f]["patient_id"]) <= ids

def expected_egfr(creatinine, age, sex):     # written separately from the generator on purpose
    k, a = (0.7, -0.241) if sex == "Female" else (0.9, -0.302)
    s = creatinine / k
    return 142 * min(s, 1) ** a * max(s, 1) ** -1.2 * 0.9938 ** age * (1.012 if sex == "Female" else 1)

def test_fev1_fvc_ratio_is_computed_from_fev1_and_fvc(sample):
    v = sample["visits"].dropna(subset=["fev1", "fvc"])
    assert len(v) > 0
    assert ((v["fev1_fvc"] - (v["fev1"] / v["fvc"] * 100).round(1)).abs() <= 0.05).all()

def test_egfr_is_computed_from_creatinine_age_and_sex(sample):
    v = sample["visits"].merge(sample["patients"][["patient_id", "sex"]], on="patient_id").dropna(subset=["creatinine"])
    assert len(v) > 0
    expected = [min(140, max(5, round(expected_egfr(c, a, s)))) for c, a, s in zip(v["creatinine"], v["age"], v["sex"])]
    assert (v["egfr"].values == np.array(expected)).all()

def test_chronic_depression_keeps_phq9_high_years_later():
    cfg = small_cfg(missing={"optional_lab": 0.0, "hospital_only": 0.0})
    p = make_patient(3, np.random.default_rng(4), cfg)
    p["dx"]["depression"] = 2010
    p["neuro_chronic"] = True
    assert make_visit_values(p, date(2025, 3, 1), np.random.default_rng(1), cfg)["phq9"] >= 10

def test_recovered_depression_has_phq9_below_10_years_later():
    cfg = small_cfg(missing={"optional_lab": 0.0, "hospital_only": 0.0})
    p = make_patient(3, np.random.default_rng(4), cfg)
    p["dx"]["depression"] = 2010
    p["neuro_chronic"] = False
    assert make_visit_values(p, date(2025, 3, 1), np.random.default_rng(1), cfg)["phq9"] < 10

def test_about_half_of_patients_get_a_chronic_course():
    shares = np.mean([make_patient(i, np.random.default_rng(i), CFG)["neuro_chronic"] for i in range(400)])
    assert abs(shares - AD["neuro_chronic_share"]) < 0.08

def test_hypertension_chance_rises_with_age_and_uses_the_scale():
    assert hypertension_chance(30) < hypertension_chance(60)
    old = AD["htn_scale"]
    try:
        AD["htn_scale"] = 0.5
        half = hypertension_chance(60)
        AD["htn_scale"] = 1.0
        assert half == pytest.approx(hypertension_chance(60) * 0.5)
    finally:
        AD["htn_scale"] = old

# ---------- undiagnosed type 2 diabetes ----------
def _t2d_patient(diagnosed):
    cfg = small_cfg(missing={"optional_lab": 0.0, "hospital_only": 0.0})
    p = make_patient(4, np.random.default_rng(6), cfg)
    p["dx"]["T2D"] = 2015
    p["t2d_diagnosed"] = diagnosed
    return p, cfg

def test_undiagnosed_t2d_raises_hba1c_but_not_has_t2d():
    p, cfg = _t2d_patient(diagnosed=False)
    d = date(2024, 3, 1)
    assert make_visit_values(p, d, np.random.default_rng(1), cfg)["hba1c"] >= 6.5
    assert has_flags(p, d)["has_T2D"] == 0 and has_flags(p, d)["has_T2D_any"] == 1

def test_diagnosed_t2d_sets_both_flags():
    p, _ = _t2d_patient(diagnosed=True)
    f = has_flags(p, date(2024, 3, 1))
    assert f["has_T2D"] == 1 and f["has_T2D_any"] == 1

def test_undiagnosed_t2d_is_labelled_1_when_two_tests_show_it():
    p, cfg = _t2d_patient(diagnosed=False)
    p["dx"]["T2D"] = 2024
    visits = [(date(2021, 5, 1), {"hba1c": 5.6, "fpg": 95}), (date(2025, 3, 1), {"hba1c": 7.1, "fpg": 140})]
    rows = {r["domain"]: r["label"] for r in training_rows(p, date(2021, 5, 1), cfg, visits=visits)}
    assert rows["T2D"] == 1

def test_fhrs_counts_only_relatives_whose_diabetes_was_diagnosed():
    fam = [{"degree": 1, "sex": "F", "conditions": [
        {"disease": "T2D", "year": 2000, "age": 50, "confirmed": True, "diagnosed": False},
        {"disease": "CVD", "year": 2001, "age": 51, "confirmed": True}]}]
    known = known_family(fam)
    assert [c["disease"] for c in known[0]["conditions"]] == ["CVD"]
    assert len(fam[0]["conditions"]) == 2                      # the full family is not changed

def test_about_the_set_share_of_relatives_diabetes_is_diagnosed():
    flags = [c["diagnosed"] for i in range(300)
             for r in make_patient(i, np.random.default_rng([7, i]), CFG)["relatives"]
             for c in r["conditions"] if c["disease"] == "T2D"]
    assert len(flags) > 100 and abs(np.mean(flags) - AD["t2d_diagnosed_share"]) < 0.08

def test_generator_writes_diagnosed_and_any_columns(sample):
    v, r = sample["visits"], sample["relatives"]
    assert "has_T2D_any" in v and "diagnosed" in r and "t2d_diagnosed" in sample["patients"]
    assert (v["has_T2D"] <= v["has_T2D_any"]).all()

# ---------- Phase 0: no label leak (docs/synthetic_data_changes.md) ----------
def _full_cfg():
    return small_cfg(missing={"optional_lab": 0.0, "hospital_only": 0.0})

def _values(p, d, seed=1):
    return make_visit_values(p, d, np.random.default_rng(seed), _full_cfg())

def test_values_before_diagnosis_do_not_depend_on_the_diagnosis_year():
    # THE leak test: two copies of one patient, only their own future diagnosis differs
    for disease in ("CVD", "T2D", "asthma", "COPD", "depression", "GAD", "HTN"):
        p = make_patient(11, np.random.default_rng(11), _full_cfg())
        sick, well = copy.deepcopy(p), copy.deepcopy(p)
        for q in (sick, well):
            q["dx"] = {k: None for k in q["dx"]}
        sick["dx"][disease] = 2025
        for d in (date(2021, 3, 1), date(2023, 3, 1), date(2025, 2, 1)):
            assert _values(sick, d) == _values(well, d), (disease, d)

def test_values_still_jump_to_high_after_diagnosis():
    p = make_patient(12, np.random.default_rng(12), _full_cfg())
    p["dx"] = {k: None for k in p["dx"]}
    p["dx"]["T2D"] = 2022
    assert _values(p, date(2021, 3, 1))["hba1c"] < 6.5 <= _values(p, date(2024, 3, 1))["hba1c"]

def test_higher_personal_risk_means_at_least_as_much_drift():
    p = make_patient(13, np.random.default_rng(13), _full_cfg())
    p["dx"] = {k: None for k in p["dx"]}
    low, high = copy.deepcopy(p), copy.deepcopy(p)
    low["risk_mult"] = {k: 0.2 for k in p["risk_mult"]}
    high["risk_mult"] = {k: 5.0 for k in p["risk_mult"]}
    d = date(2024, 3, 1)
    zl, zh = risk_z(low, d), risk_z(high, d)
    for name in ("hba1c", "fpg", "sbp", "phq9", "spo2", "bmi"):
        assert drift_position(high, name, d, zh) >= drift_position(low, name, d, zl), name

def _extreme_risk_patient(seed):
    p = make_patient(seed, np.random.default_rng(seed), _full_cfg())
    p["dx"] = {k: None for k in p["dx"]}
    p["risk_mult"] = {k: 50.0 for k in p["risk_mult"]}
    p["speed"] = 10.0
    p["base_mod"] = {k: True for k in p["base_mod"]}
    return p

def test_label_defining_values_never_meet_their_rule_before_diagnosis():
    for seed in range(14, 24):
        p = _extreme_risk_patient(seed)
        for d in (date(2021, 3, 1), date(2026, 3, 1)):
            v = _values(p, d, seed)
            assert v["hba1c"] < 6.5 and v["fpg"] < 126 and v["sbp"] < 140 and v["dbp"] < 90, (seed, d)
            assert v["phq9"] < 10 and v["gad7"] < 10 and v["fev1_fvc"] >= 70 and v["pg_2h"] < 200, (seed, d)
            assert all(abnormal_tests(dom, v) == 0 for dom in APP_DOMAINS), (seed, d)

def test_the_cap_bands_are_below_every_rule_and_the_next_band_is_not():
    # FEV1/FVC: the whole Moderate band (60-69.9) meets the CRD rule, so the cap is the Low band
    assert SPEC["fev1_fvc"]["low"][0] >= 70 and SPEC["fev1_fvc"]["mod"][1] < 70
    assert SPEC["hba1c"]["mod"][1] < 6.5 and SPEC["fpg"]["mod"][1] < 126 and SPEC["pg_2h"]["mod"][1] < 200
    assert SPEC["sbp"]["mod"][1] < 140 and SPEC["dbp"]["mod"][1] < 90
    assert SPEC["phq9"]["mod"][1] < 10 and SPEC["gad7"]["mod"][1] < 10

def test_values_that_define_no_label_can_reach_high_before_diagnosis():
    p = _extreme_risk_patient(14)
    v = _values(p, date(2026, 3, 1))
    assert v["bmi"] >= 25 and v["waist_cm"] >= 102 and v["heart_rate"] >= 100

def test_high_values_do_not_pile_up_at_the_band_maximum():
    # past the start of High the position nears the top gradually: 0.5 band past High = 39% of the band, 1 band = 63%
    p = make_patient(26, np.random.default_rng(26), _full_cfg())
    p["dx"] = {k: None for k in p["dx"]}
    p["base_mod"] = {k: True for k in p["base_mod"]}
    p["risk_mult"] = {k: 5.0 for k in p["risk_mult"]}
    d = date(2026, 3, 1)
    z, fracs = risk_z(p, d), []
    for speed in (2.0, 3.0, 4.0, 6.0):
        p["speed"] = speed
        lvl, frac = band_and_u(p, "waist_cm", d, z)
        if lvl == "high":
            pos = drift_position(p, "waist_cm", d, z)
            assert frac == pytest.approx(1 - np.exp(-(pos - 2)))
            fracs.append(frac)
    assert fracs and fracs == sorted(fracs) and max(fracs) < 0.99

def test_causes_of_a_disease_are_not_forced_high_by_its_diagnosis():
    p = make_patient(27, np.random.default_rng(27), _full_cfg())
    p["dx"] = {k: None for k in p["dx"]}
    ill = copy.deepcopy(p)
    ill["dx"].update({"T2D": 2015, "CVD": 2016, "HTN": 2016})
    d = date(2024, 3, 1)
    a, b = _values(p, d), _values(ill, d)
    for name in NOT_FORCED_AFTER_DX:
        assert a[name] == b[name], name                                  # same path with or without the diagnosis
    assert a["weight_kg"] == b["weight_kg"]
    assert b["hba1c"] >= 6.5 and b["sbp"] >= 140                         # consequences still go High

def test_lower_is_worse_values_go_down_as_risk_rises():
    p = make_patient(25, np.random.default_rng(25), _full_cfg())
    p["dx"] = {k: None for k in p["dx"]}
    p["base_mod"] = {k: False for k in p["base_mod"]}
    d, out = date(2024, 3, 1), []
    for m in (0.2, 1.0, 5.0, 25.0):
        q = copy.deepcopy(p)
        q["risk_mult"] = {k: m for k in p["risk_mult"]}
        out.append(_values(q, d, seed=3))
    for name in ("hdl", "spo2", "fev1_pct_pred", "pef", "sleep_hours"):      # fev1_fvc: recomputed, rounding
        seq = [v[name] for v in out]
        assert all(b <= a for a, b in zip(seq, seq[1:])), (name, seq)
    assert [v["sbp"] for v in out] == sorted(v["sbp"] for v in out)

def test_lifestyle_values_do_not_drift_with_risk():
    p = make_patient(15, np.random.default_rng(15), _full_cfg())
    d = date(2024, 3, 1)
    low, high = copy.deepcopy(p), copy.deepcopy(p)
    low["risk_mult"] = {k: 0.2 for k in p["risk_mult"]}
    high["risk_mult"] = {k: 5.0 for k in p["risk_mult"]}
    for name in P0["lifestyle_values"]:
        assert drift_position(low, name, d, risk_z(low, d)) == drift_position(high, name, d, risk_z(high, d))

def test_personal_multiplier_averages_1_so_calibrated_rates_keep_their_meaning():
    rng = np.random.default_rng(16)
    for sex in ("M", "F"):
        sh = factor_shares(sex)
        draws = []
        for _ in range(20000):
            s = rng.random()
            f = {k: int(rng.random() < q) for k, q in sh.items() if k not in ("smoker_current", "smoker_former")}
            f["smoker_current"] = int(s < sh["smoker_current"])
            f["smoker_former"] = int(sh["smoker_current"] <= s < sh["smoker_current"] + sh["smoker_former"])
            draws.append(personal_multipliers(f, sex, rng))
        for dis in RISK_DISEASES:
            assert abs(np.mean([m[dis] for m in draws]) - 1) < 0.06, (sex, dis)

def test_flags_are_drawn_from_their_new_shares_not_from_a_future_diagnosis():
    pts = [make_patient(i, np.random.default_rng([17, i]), CFG) for i in range(400)]
    women = [q for q in pts if q["sex"] == "F"]
    assert all(q["gestational_diabetes"] == 0 for q in pts if q["sex"] == "M")
    no_asthma_before = [q["allergic_rhinitis"] for q in pts if not began_before_window(q, "asthma")]
    assert abs(np.mean(no_asthma_before) - P0["allergic_rhinitis_share"]) < 0.06
    assert np.mean([q["gestational_diabetes"] for q in women]) < 0.10

def test_acanthosis_chance_ignores_a_future_diabetes_diagnosis():
    p = make_patient(19, np.random.default_rng(19), CFG)
    later, never = copy.deepcopy(p), copy.deepcopy(p)
    later["dx"]["T2D"], never["dx"]["T2D"] = 2024, None
    assert acanthosis_chance(later) == acanthosis_chance(never)
    before = copy.deepcopy(p)
    before["dx"]["T2D"] = 2015
    assert acanthosis_chance(before) == P0["acanthosis_share_if_t2d_before_window"]

def test_rhinitis_is_added_only_for_asthma_that_began_before_the_window():
    added = 0
    for i in range(400):
        p = make_patient(i, np.random.default_rng([20, i]), CFG)
        if began_before_window(p, "asthma"):
            added += p["allergic_rhinitis"] - p["allergic_rhinitis_before_disease"]
        else:                                                  # future or no asthma: the pre-disease draw only
            assert p["allergic_rhinitis"] == p["allergic_rhinitis_before_disease"], i
    assert added > 0

def test_bmi_uses_asian_cutoffs():
    assert BMI_BANDS == {"low": (18.5, 22.9), "mod": (23.0, 24.9), "high": (25.0, 40.0)}

def test_hypertension_hazard_rebuilds_the_approved_prevalence_curve():
    for age in (30, 45, 60, 75):
        free = (1 - htn_hazard(HTN_START_AGE)) * np.prod([1 - htn_hazard(a) for a in range(HTN_START_AGE + 1, age + 1)])
        assert 1 - free == pytest.approx(hypertension_chance(age), abs=1e-9)

# ---------- CVD label (physician decision, Nov 5 2026) ----------
def _no_disease_patient():
    p = make_patient(18, np.random.default_rng(18), CFG)
    p["dx"] = {k: None for k in p["dx"]}
    return p

def _label(p, domain, index, visits, col="label_recorded"):
    return {r["domain"]: r[col] for r in training_rows(p, index, CFG, visits=visits)}.get(domain, "excluded")

def _bp(d, sbp, dbp):
    return (d, {"sbp": sbp, "dbp": dbp})

def _cvd_label(p, index, visits):
    return _label(p, "CVD", index, visits)

def test_cvd_label_high_bp_on_two_visits_counts():
    p = _no_disease_patient()
    p["dx"]["HTN"] = 2022
    visits = [_bp(date(2021, 3, 1), 125, 80), _bp(date(2022, 9, 1), 150, 85), _bp(date(2022, 11, 1), 138, 92)]
    assert _cvd_label(p, date(2021, 3, 1), visits) == 1

def test_cvd_label_one_high_bp_visit_is_not_enough():
    p = _no_disease_patient()
    visits = [_bp(date(2021, 3, 1), 125, 80), _bp(date(2022, 9, 1), 150, 85), _bp(date(2023, 9, 1), 130, 85)]
    assert _cvd_label(p, date(2021, 3, 1), visits) == 0

def test_cvd_label_second_high_visit_after_5_years_is_0():
    p = _no_disease_patient()
    visits = [_bp(date(2021, 3, 1), 125, 80), _bp(date(2025, 9, 1), 150, 85), _bp(date(2026, 6, 1), 150, 95)]
    assert _cvd_label(p, date(2021, 3, 1), visits) == 0

def test_cvd_event_alone_counts():
    p = _no_disease_patient()
    p["dx"]["CVD"] = 2024
    assert _cvd_label(p, date(2021, 3, 1), [_bp(date(2021, 3, 1), 120, 80)]) == 1

def test_hypertension_at_index_means_already_ill():
    p = _no_disease_patient()
    p["dx"]["HTN"] = 2015
    assert _cvd_label(p, date(2021, 3, 1), [_bp(date(2021, 3, 1), 150, 95)]) == "excluded"

def test_high_bp_at_the_index_visit_means_already_ill():
    p = _no_disease_patient()
    assert _cvd_label(p, date(2021, 3, 1), [_bp(date(2021, 3, 1), 150, 80)]) == "excluded"

# ---------- T2D, CRD, NEURO labels (physician-approved rules, Nov 5 2026) ----------
I = date(2021, 3, 1)

def test_t2d_hba1c_and_fpg_from_one_sample_are_two_tests():
    p = _no_disease_patient()
    assert _label(p, "T2D", I, [(I, {"hba1c": 5.5, "fpg": 90}), (date(2024, 1, 1), {"hba1c": 6.8, "fpg": 131})]) == 1

def test_t2d_two_tests_on_two_visits_count():
    p = _no_disease_patient()
    visits = [(I, {"hba1c": 5.5}), (date(2023, 1, 1), {"hba1c": 6.6, "fpg": np.nan}), (date(2024, 1, 1), {"fpg": 127})]
    assert _label(p, "T2D", I, visits) == 1

def test_t2d_one_abnormal_test_is_not_enough():
    p = _no_disease_patient()
    assert _label(p, "T2D", I, [(I, {"hba1c": 5.5}), (date(2024, 1, 1), {"hba1c": 6.9, "fpg": 110})]) == 0

def test_t2d_true_onset_without_any_test_is_label_0():
    p = _no_disease_patient()
    p["dx"]["T2D"] = 2023
    assert _label(p, "T2D", I, [(I, {"hba1c": np.nan, "fpg": np.nan})]) == 0

def test_undiagnosed_t2d_with_high_hba1c_at_index_is_left_out():
    p = _no_disease_patient()
    p["t2d_diagnosed"] = False
    assert _label(p, "T2D", I, [(I, {"hba1c": 6.7, "fpg": 110})]) == "excluded"
    assert _label(p, "T2D", I, [(I, {"hba1c": 5.9, "fpg": 130})]) == "excluded"

def test_undiagnosed_t2d_that_began_before_index_is_left_out_even_without_labs():
    p = _no_disease_patient()
    p["dx"]["T2D"], p["t2d_diagnosed"] = 2019, False
    assert _label(p, "T2D", I, [(I, {"hba1c": np.nan, "fpg": np.nan})]) == "excluded"

def test_crd_one_low_fev1_fvc_counts():
    p = _no_disease_patient()
    assert _label(p, "CRD", I, [(I, {"fev1_fvc": 78.0}), (date(2024, 1, 1), {"fev1_fvc": 66.0})]) == 1

def test_crd_asthma_diagnosis_counts_without_spirometry():
    p = _no_disease_patient()
    p["dx"]["asthma"] = 2023
    assert _label(p, "CRD", I, [(I, {"fev1_fvc": np.nan})]) == 1

def test_crd_copd_without_spirometry_is_label_0():
    p = _no_disease_patient()
    p["dx"]["COPD"] = 2023
    assert _label(p, "CRD", I, [(I, {"fev1_fvc": np.nan}), (date(2024, 1, 1), {"fev1_fvc": np.nan})]) == 0

def test_crd_low_fev1_fvc_at_index_is_left_out():
    p = _no_disease_patient()
    assert _label(p, "CRD", I, [(I, {"fev1_fvc": 65.0})]) == "excluded"

def test_neuro_two_visits_count_and_one_does_not():
    p = _no_disease_patient()
    two = [(I, {"phq9": 4, "gad7": 3}), (date(2023, 1, 1), {"phq9": 12, "gad7": 3}), (date(2023, 3, 1), {"phq9": 6, "gad7": 11})]
    one = [(I, {"phq9": 4, "gad7": 3}), (date(2023, 1, 1), {"phq9": 12, "gad7": 3}), (date(2023, 3, 1), {"phq9": 6, "gad7": 5})]
    assert _label(p, "NEURO", I, two) == 1 and _label(p, "NEURO", I, one) == 0

def test_neuro_high_score_at_index_is_left_out():
    p = _no_disease_patient()
    assert _label(p, "NEURO", I, [(I, {"phq9": 11, "gad7": 2})]) == "excluded"

def test_no_training_row_meets_its_rule_at_the_index_visit(sample):
    first = sample["visits"][sample["visits"]["visit_no"] == 1].set_index("patient_id")
    for r in sample["training"].itertuples():
        assert abnormal_tests(r.domain, first.loc[r.patient_id].to_dict()) == 0, (r.patient_id, r.domain)

def test_label_follows_the_true_onset_even_without_any_test():
    p = _no_disease_patient()
    p["dx"]["T2D"], p["t2d_diagnosed"] = 2023, False
    visits = [(I, {"hba1c": np.nan, "fpg": np.nan})]
    assert _label(p, "T2D", I, visits, "label") == 1 and _label(p, "T2D", I, visits) == 0

def test_label_is_0_when_the_onset_is_after_5_years():
    p = _no_disease_patient()
    p["dx"]["COPD"] = 2027
    assert _label(p, "CRD", I, [(I, {"fev1_fvc": 80.0})], "label") == 0

def test_cvd_true_label_counts_hypertension_onset():
    p = _no_disease_patient()
    p["dx"]["HTN"] = 2023
    assert _label(p, "CVD", I, [_bp(I, 120, 80)], "label") == 1

def test_every_generated_label_matches_the_true_onset(sample):
    pts = sample["patients"].set_index("patient_id")
    for r in sample["training"].itertuples():
        cols = [f"true_dx_year_{d}" for d in APP_DOMAINS[r.domain]] + (["true_dx_year_HTN"] if r.domain == "CVD" else [])
        years = [int(y) for y in pts.loc[r.patient_id, cols] if pd.notna(y)]
        idx = pd.Timestamp(r.index_date)
        onset = min((pd.Timestamp(y, 7, 1) for y in years), default=None)
        assert onset is None or onset > idx, (r.patient_id, r.domain)          # already ill -> no row
        expected = int(onset is not None and onset <= idx + pd.DateOffset(years=CFG["label_window_years"]))
        assert r.label == expected, (r.patient_id, r.domain)

def test_a_recorded_label_never_comes_without_a_true_onset(sample):
    t = sample["training"]
    assert (t["label_recorded"] <= t["label"]).all()

def test_bmi_value_follows_the_calibrated_parts():
    for cal in BMI_CAL.values():
        assert bmi_value("low", 0.0, cal) == (BMI_UNDERWEIGHT[0] if cal["underweight_of_low"] > 0 else BMI_BANDS["low"][0])
        assert bmi_value("low", 1.0, cal) == BMI_BANDS["low"][1] and bmi_value("mod", 0.0, cal) == BMI_BANDS["mod"][0]
        assert bmi_value("high", 0.0, cal) == 25.0 and bmi_value("high", 1.0, cal) == 40.0
        for lvl in ("low", "mod", "high"):
            vals = [bmi_value(lvl, f, cal) for f in np.linspace(0, 1, 101)]
            assert vals == sorted(vals), lvl
        below_30 = np.mean([bmi_value("high", f, cal) < 30 for f in np.linspace(0, 0.999, 1000)])
        assert below_30 == pytest.approx(1 - cal["obese30_of_high"], abs=0.01)
        under = np.mean([bmi_value("low", f, cal) < 18.5 for f in np.linspace(0, 0.999, 1000)])
        assert under == pytest.approx(cal["underweight_of_low"], abs=0.01)

def test_bmi_starting_band_shares_by_age_group():
    pts = [make_patient(i, np.random.default_rng([21, i]), CFG) for i in range(2000)]
    for group, cal in BMI_CAL.items():
        mine = [q for q in pts if q["bmi_group"] == group]
        bands = np.array([bmi_start_band(q) for q in mine])
        assert len(mine) > 200, group
        assert abs(np.mean(bands == 2) - cal["start_high"]) < 0.05, group
        assert abs(np.mean(bands == 1) - cal["start_mod"]) < 0.05, group
        ages = [CFG["window"]["start_year"] - q["birth_year"] for q in mine]
        assert all((a >= BMI_OLDER_FROM) == (group == "60+") for a in ages)

def test_every_generated_label_matches_the_rule_on_the_recorded_visits(sample):
    v = sample["visits"].assign(visit_date=lambda x: pd.to_datetime(x["visit_date"])).sort_values(["patient_id", "visit_date"])
    pts = sample["patients"].set_index("patient_id")
    for r in sample["training"].itertuples():
        mine = v[v["patient_id"] == r.patient_id]
        idx = pd.Timestamp(r.index_date)
        end = idx + pd.DateOffset(years=CFG["label_window_years"])
        later = mine[(mine["visit_date"] > idx) & (mine["visit_date"] <= end)]
        hits = np.cumsum([abnormal_tests(r.domain, row) for row in later.to_dict("records")])
        by_rule = bool(len(hits) and hits[-1] >= RULE_TIMES[r.domain])
        routes = [pts.loc[r.patient_id, f"true_dx_year_{d}"] for d in RULE_ROUTE[r.domain]]
        by_route = any(pd.notna(y) and idx < pd.Timestamp(int(y), 7, 1) <= end for y in routes)
        assert r.label_recorded == int(by_rule or by_route), (r.patient_id, r.domain)

def test_has_cvd_is_event_or_hypertension(sample):
    v = sample["visits"]
    assert ((v["has_CVD"] == 1) == ((v["has_CV_event"] == 1) | (v["has_HTN"] == 1))).all()
    assert v["has_HTN"].sum() > 0

.

.

.

.

.

.

... (57 outputs left out)

.

.

                            [100%]


65 passed in 31.92s


In [26]:
# Cell 3 - RUN  (final run, Nov 6: 20,000 patients and two seeds, about 8 minutes)
# Quick test: start Jupyter with SYNTH_N=5000 in the environment.
import os
N_PATIENTS = int(os.environ.get("SYNTH_N", 20000))
SEED2 = 20261102                    # second seed (author-defined): the external check in train_models.ipynb
OUT = generate(CFG, n=N_PATIENTS, out_dir="output")
OUT2 = generate({**CFG, "seed": SEED2}, n=N_PATIENTS, out_dir="output_seed2")
for folder, out in (("output", OUT), ("output_seed2", OUT2)):
    for name, df in out.items():
        print(f"{folder}/{name}.csv: {len(df):,} rows, {df.shape[1]} columns")


output/patients.csv: 20,000 rows, 27 columns
output/relatives.csv: 242,876 rows, 12 columns
output/visits.csv: 131,953 rows, 79 columns
output/training.csv: 70,297 rows, 5 columns
output_seed2/patients.csv: 20,000 rows, 27 columns
output_seed2/relatives.csv: 243,975 rows, 12 columns
output_seed2/visits.csv: 132,261 rows, 79 columns
output_seed2/training.csv: 70,071 rows, 5 columns
